# Due Diligence MAS - Colab notebook

Multi-agent system that produces a startup due diligence memo. The same workflow is implemented in **CrewAI** and in **LangGraph**.

**This notebook is an extract.** The full project lives in the GitHub repo **[gmonitillodev-hub/Due-Diligence-MAS](https://github.com/gmonitillodev-hub/Due-Diligence-MAS)**. Read the repo for details: README, design choices, tests, benchmarks.

Input: startup name, website, sector, additional info. Output: a Markdown memo.

| Section | Content |
| --- | --- |
| 1. Variables | Install packages, load secrets, set config |
| 2. Utils and shared | Constants, classes, LLM provider, tools, agents (used by both frameworks) |
| 3. MCP | Task manager MCP server and its tool wiring |
| 4. CrewAI | Crews and flow |
| 5. LangGraph | Nodes and graph |
| 6. Servers | FastAPI gateways and MCP HTTP server, then a demo run |

**Colab limits that shape this notebook**
- The code is written to files with `%%writefile` and imported as the package `due_diligence_mas`. The MCP server starts as a stdio subprocess and needs a real module.
- Colab opens no public port. The servers listen on `127.0.0.1` and you call them from the notebook.
- Files in `/content` are lost when the runtime recycles. The DB, logs and tasks are not persistent.
- Secrets come from the Colab **Secrets** panel (key icon on the left). Enable *Notebook access* for each one.
- A full run makes many LLM and search calls. It takes minutes and costs money.

**How to use**
1. Add the Colab secrets: `OPENAI_API_KEY` (or `GROQ_API_KEY` if `LLM_PROVIDER = "groq"`) and `TAVILY_API_KEY`. Optional: `GITHUB_TOKEN`, `LANGSMITH_API_KEY`. Missing required keys are asked with a prompt.
2. Run the cells in order, top to bottom: setup, shared code, MCP, the framework code, then the servers.
3. In section 6 choose the framework (`langgraph` or `crewai`) and enter the company data. The run starts and `wait()` prints each status change.
4. At `awaiting_review` the analysis preview is shown. Answer `proceed` (run the debate, then the memo) or `end` (write the memo from the analysis only).
5. At `completed` the final memo is shown as Markdown. At `failed` check the server log in `/content/Due-Diligence-MAS/log/`.
6. Run the last cell to stop the servers.

**Workflow**: company profiler -> analysts (commercial, team, tech) -> analysis review (creates follow-up tasks through MCP) -> human review -> risk/opportunity debate and judge -> memo writer.

**Run statuses**: `running`, `awaiting_review`, `completed`, `failed`.


## Project overview

Multi-agent system that produces a startup due diligence memo. It implements the same workflow in two frameworks, **CrewAI** and **LangGraph**, to compare them.

**Input:** startup name, website, sector, additional info → **Output:** a Markdown memo with profile, specialist analyses and, optionally, debate findings.

1. **Profiling** — company profile and information gaps.
2. **Analysis** — Tech, Commercial and Team specialists run in parallel, then one aggregation.
3. **Analysis review** — a reviewer scores the analysis (0–10) and may create follow-up tasks through an MCP server.
4. **Human review** — the user chooses `proceed` (run Debate) or `end` (skip it).
5. **Debate** (optional) — Pro vs. Contra extractors, judged.
6. **Memo** — final formatted memo.

| Aspect | Choice |
| --- | --- |
| Orchestration patterns | Debate/adversarial, Supervisor, Sequential |
| Agentic paradigms | ReAct; Reflexion (LangGraph Supervisor) |
| CrewAI | Version 2 under test; Version 1 discarded ([open points](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/open_points.md)) |
| Persistence | LangGraph: SQLite; CrewAI: SQLite |

Full documentation in the repo: [Architecture](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/architecture.md), [Open points](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/open_points.md), [Runtime](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/runtime.md), [REST API](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/rest_api.md), [MCP Server](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/mcp_task_server.md), [Benchmark](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/benchmark_registry.md).

## Architecture

### Shared Agent Architecture

Each agent is a class that inherits from `BaseAgent`. Its description and task instructions live in shared constants, so CrewAI and LangGraph use the same prompt and task definition. A framework adapter maps them to the runtime inputs.

| Shared definition | CrewAI adapter | LangGraph adapter |
| --- | --- | --- |
| Agent description | Agent description and backstory | System prompt |
| Task instructions | CrewAI task | Human message |

### Orchestration Patterns and Agentic Paradigms

| Category | Implemented |
| --- | --- |
| Orchestration patterns | **Debate / adversarial** (Debate module: Pro vs. Contra, judged); **Supervisor** (manager or Supervisor node); **Sequential** (crew and node chaining) |
| Agentic paradigms | **ReAct** (basic, for agents using tools); **Reflexion** (the Supervisor in LangGraph reviews profile and analysis and feeds back to improve them) |

### Common Pipeline

Both frameworks follow the same logical steps:

1. **Profiling** — build the company profile and record missing information.
2. **Analysis** — Tech, Commercial and Team specialists, then one aggregation.
3. **Analysis review** — a reviewer gives the aggregated analysis a completeness score (0–10), lists missing items and may create follow-up tasks through the task manager MCP tools.
4. **Human review** — the user sees the analysis and the score, then chooses `proceed` or `end`.
5. **Debate** (only on `proceed`) — Formatter, Pro Extractor, Contra Extractor, Judge, in sequence.
6. **Memo Writer** — one agent, one task: the final formatted memo.

The score informs the human; it does not decide the route. An automatic gate on the score is not implemented (see [OP-04](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/open_points.md#op-04--score-based-debate-gating)).

### CrewAI — Version 1 (discarded)

Version 1 is not part of the current implementation. It differs from Version 2 (AS IS) in two ways:

- **Crews:** three (Analysis, Debate, Memo Writer) instead of five. Profiling is not a separate crew: it is part of one hierarchical Analysis Crew.
- **Manager:** a manager coordinates the Company Profiler and the Tech, Team and Commercial agents, and chooses the context each one receives. In Version 2 the manager only checks the profiling quality.

It was discarded because of two problems that were both confirmed:

1. **Overlapping delegation (OP-01).** With the hierarchical process and asynchronous specialist tasks, the manager executes every task, so the async tasks call the same executor in parallel. The run failed with `Executor is already running. Cannot invoke the same executor instance concurrently.` (CrewAI 1.15.41, `openai/gpt-4o-mini`). Running the specialists tasks sequentially (`async_execution=false`) avoids it, but gives up the parallelism.
2. **Non-deterministic context handoff (OP-02).** A task should receive the full output of its declared context, but the manager passes an extreme summary, so specialists repeat retrieval calls. Prompt instructions do not make the transfer deterministic.

Version 2 removes the manager from the analysis and passes the context explicitly.

### CrewAI — Version 2

Version 2 removes the manager from the analysis. Profiling becomes a separate crew and the specialists run as explicit asynchronous tasks.

| Order | Crew | Process | Output |
| --- | --- | --- | --- |
| 1 | Profiling | Hierarchical: manager and Company Profiler | Company profile and information gaps |
| 2 | Analysis | Sequential: three async specialist tasks, then one synchronous aggregation task | Consolidated analysis |
| 3 | Analysis Review | One agent (the Supervisor as reviewer), one task | Score (0–10), missing items, summary, optional follow-up tasks |
| 4 | Debate | Sequential | Debate findings and judgment |
| 5 | Memo Writer | One agent, one task | Final memo |

```mermaid
flowchart TD
    Input["User input"] --> Manager
    subgraph Profiling["Profiling Crew — hierarchical"]
        Manager["Manager — quality review"]
        Profiler["Company Profiler"]
        Manager -->|"Delegate / bounded retry"| Profiler
        Profiler -->|"Profile and gaps"| Manager
    end
    Manager --> Profile["Company profile"]
    subgraph Analysis["Analysis Crew — sequential"]
        Tech["Tech — async"]
        Commercial["Commercial — async"]
        Team["Team — async"]
        Aggregate["Aggregation — sync"]
        Tech --> Aggregate
        Commercial --> Aggregate
        Team --> Aggregate
    end
    Profile -->|"General + tech context"| Tech
    Profile -->|"General + commercial context"| Commercial
    Profile -->|"General + team context"| Team
    Profile --> Aggregate
    Aggregate --> Report["Consolidated analysis"]
    Report --> Reviewer["Analysis Review Crew — score 0–10"]
    Reviewer -->|"Gaps"| Tasks["Follow-up tasks (MCP)"]
    Reviewer --> Human{"Human review"}
    Human -->|"end"| Writer["Memo Writer Crew"]
    Human -->|"proceed"| Debate["Debate Crew"]
    Debate --> Writer
    Writer --> Memo["Formatted memo"]
```

#### Profiling Crew

The manager guarantees the profile quality. The Profiler gathers information with search and retrieval tools. If information is missing, the manager can ask for another iteration. A fixed maximum iteration count bounds the loop, and prompt instructions discourage repeating the same request. Remaining gaps are recorded as missing information.

#### Analysis Crew

Tech, Commercial and Team run in parallel. Each receives the general profile plus only its own specialist context, which reduces noise and unnecessary agent loops. A Memo Writer agent then waits for all three and aggregates them with the profile into one analysis. This aggregation is separate from the final memo.

#### Analysis Review Crew

A single reviewer scores the whole aggregated analysis from 0 (nothing usable) to 10 (every area covered and source-backed). It checks topic coverage, grounding in the profile, cited sources and explicit gaps. It returns a score, a list of missing items and a short summary.

If gaps need human work, the reviewer creates follow-up tasks through the task manager MCP tools (see [MCP Server](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/mcp_task_server.md)): at most one per topic (tech, commercial), assigned by role, with priority HIGH, MEDIUM or LOW. Team gaps are only mentioned in the review.

#### Debate Crew

The four agents run in this order: Formatter, Pro Extractor, Contra Extractor, Judge.

#### Memo Writer Crew

One agent, one task. It consolidates the profile and specialist analyses, plus the Debate output when Debate has run, into the final memo.

#### Human Review

After the Analysis Review the Flow shows the analysis preview and the score and asks whether to proceed.

| User decision | Route | Result |
| --- | --- | --- |
| `end` | Memo Writer directly | Memo from profile and analyses, no Debate cost |
| `proceed` | Debate, then Memo Writer | Memo including debate findings |

Skipping Debate saves its tokens and time; the memo is still generated.

#### LangGraph

A single graph orchestrates the workflow and reuses the shared agent classes. The Supervisor is a project-defined node with a code-based router; it does not use `create_supervisor` or LLM-driven routing.

| Node | Responsibility |
| --- | --- |
| Request Handler | Asks the user for company information. Skipped when the input is already complete (for example a full REST request). |
| Supervisor and router | Calls the Profiler, reviews the profile and returns gaps and feedback. Proceeds to Fan-out when the profile is complete or after three passes in total. |
| Profiler | Produces the profile, using the Supervisor feedback on later passes. |
| Fan-out | Uses `Send` to run Tech, Commercial and Team in parallel. |
| Aggregator | Waits for the three analyses and consolidates them with the profile. Control returns to the Supervisor. |
| Supervisor review and interrupt | Scores the consolidated analysis (completeness score and review summary), then interrupts for human review. |
| Memo Writer | Formats the final memo. Both resume paths end here. |

#### Resume after Human Review

| Resume value | Route |
| --- | --- |
| `end` | Memo Writer directly |
| `proceed` | Debate, then Memo Writer |

The user's value selects the route; the score only informs the decision. The SQLite checkpointer resumes the graph with the same `thread_id`, even after a restart. See [LangGraph Persistence and Resume](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/runtime.md#langgraph-persistence-and-resume).

#### Flow Diagram — Profiling and Analysis

```mermaid
flowchart TD
    Start["Start"] --> InputReady{"Input complete?"}
    InputReady -->|"Yes"| Router["Supervisor router"]
    InputReady -->|"No"| Handler["Request Handler"]
    Handler --> Router
    Router --> Profiler["Profiler"]
    Profiler --> Supervisor["Supervisor profile review"]
    Supervisor -->|"Gaps and feedback"| Router
    Router -->|"Complete or 3 passes"| FanOut["Fan-out via Send"]
    FanOut --> Tech["Tech"]
    FanOut --> Commercial["Commercial"]
    FanOut --> Team["Team"]
    Tech --> Aggregator["Aggregator"]
    Commercial --> Aggregator
    Team --> Aggregator
```

#### Flow Diagram — Review and Memo

```mermaid
flowchart TD
    Aggregator["Aggregator"] --> Supervisor["Supervisor review — score"]
    Supervisor --> Interrupt["Interrupt: human review"]
    Interrupt --> Resume{"Resume"}
    Resume -->|"end"| Writer["Memo Writer"]
    Resume -->|"proceed"| Debate["Debate"]
    Debate --> Writer
    Writer --> Memo["Formatted memo"]
```

## Benchmark

### 1. Execution Overview

Six executions, 3 companies × 2 frameworks, all with `gpt-4o-mini`. All finished successfully. CrewAI figures refer to the first CrewAI version; Version 2 is still to be benchmarked.

| Company | Framework | Time (s) | LLM Calls | Total Tokens | Cost (USD) |
|---|---|---:|---:|---:|---:|
| Synthesia | CrewAI | 358.87 | 30 | 199,475 | $0.0376 |
| Synthesia | LangGraph | 173.83 | 23 | 173,057 | $0.0320 |
| ElevenLabs | CrewAI | 328.87 | 32 | 137,770 | $0.0282 |
| ElevenLabs | LangGraph | 248.61 | 28 | 166,337 | $0.0317 |
| Qdrant | CrewAI | 508.93 | 41 | 243,938 | $0.0468 |
| Qdrant | LangGraph | 314.62 | 29 | 193,755 | $0.0359 |

**Pending:** Wayve, Mistral AI and Cohere.

### 2. Comparison

| Metric | CrewAI (avg) | LangGraph (avg) | Δ |
|---|---:|---:|---|
| Time (s) | 398.89 | 245.69 | LangGraph ~38% faster |
| LLM Calls | 34.3 | 26.7 | LangGraph ~22% fewer |
| Total Tokens | 193,727 | 177,716 | LangGraph ~8% fewer |
| Cost (USD) | $0.0376 | $0.0332 | LangGraph ~12% cheaper |

Both frameworks use the same model and a similar architecture, so differences are small. LangGraph is faster and makes fewer calls, probably because of lower orchestration overhead and fewer internal retries. Cost differs by about $0.004 per run. Memo quality is equivalent. The framework choice depends mainly on developer experience, observability (LangSmith vs. CrewAI tracing) and ecosystem.

## Design choices

Extract from [crewai_vs_langgraph.md](https://github.com/gmonitillodev-hub/Due-Diligence-MAS/blob/main/doc/crewai_vs_langgraph.md), the framework choice and target architecture.

**Decision.** Use LangGraph to orchestrate the end-to-end workflow and CrewAI inside the Analysis and Debate nodes. This notebook keeps both full implementations as a basis for comparison.

CrewAI Flows also support routing, persistence and pause/resume after human feedback. LangGraph is preferred because the graph makes the shared state, the transitions and the resume points explicit. The criterion is **how much control is needed over data and transitions between phases**.

| Framework | Strengths | Weaknesses in the POC |
| --- | --- | --- |
| CrewAI Flows | Agent/Task/Crew fit Analysis and Debate. Bounded manager delegation. Typed state, routing, persistence and human feedback. | V1 manager delegation gave overlapping calls and over-summarized context. Data handoffs between crews are still application code. End-to-end REST resume is still to be verified. |
| LangGraph | Explicit state and transitions. Supervisor judgment separated from routing. Resume tied to `thread_id` through the checkpointer. | More implementation work (state schema, routing, reducers). A resume restarts the suspended node, so earlier operations must be safe to repeat. |

**Target architecture.** LangGraph is the orchestrator. Analysis and Debate are two nodes that each invoke a CrewAI crew. The Supervisor is a single agent invoked by the graph.

| Component | Responsibility |
| --- | --- |
| Profiler and Supervisor | The graph repeats profiling until the profile is complete or three passes in total have run. After Analysis the Supervisor returns a completeness score and a summary. |
| Analysis Crew | Tech, Team and Commercial run in parallel and are consolidated. The node returns structured contributions, missing information and the consolidated analysis. |
| Human review | The thread is suspended and the preview is exposed. Resume uses the same `thread_id` with `proceed` or `end`. The score is informational. |
| Debate Crew | Only on `proceed`: Formatter (if enabled), Pro Extractor, Contra Extractor, Judge. |
| Memo Writer | Both branches converge here. The Markdown memo includes the Debate results when present. |


## 1. Variables

In [ ]:
import sys

assert (3, 12) <= sys.version_info[:2] < (3, 14), f"Python {sys.version.split()[0]} is not supported: the project needs 3.12 or 3.13"

Install the dependencies (same ranges as `pyproject.toml`). If pip reports conflicts with preinstalled Colab packages, ignore them unless an import fails. If an import fails, use **Runtime > Restart session** and run again from the next cell.

In [ ]:
%pip install -q "crewai[litellm]>=1.15.40,<2.0.0" "crewai-tools>=1.15.41,<2.0.0" "pydantic>=2.11.9,<2.13" \
    "tavily-python>=0.8.5,<0.9.0" "python-dotenv>=1.2.4,<2.0.0" "langchain-apify>=0.2.0,<0.3.0" \
    "langchain>=1.5.0,<2.0.0" "langgraph>=1.2.22,<2.0.0" "langchain-tavily>=0.2.18,<0.3.0" \
    "langchain-openai>=1.8.0,<2.0.0" "langsmith>=0.15.0,<0.16.0" "langgraph-supervisor>=0.0.31,<0.0.32" \
    "fastapi>=0.165.0,<0.166.0" "langgraph-checkpoint-sqlite>=3.1.3,<4.0.0" "uvicorn>=0.54.0,<0.55.0" \
    "mcp>=1.30.0,<2" "langchain-mcp-adapters>=0.3.2,<0.4.0" requests

Configuration. Change the values in this cell only.

- `LLM_PROVIDER`: `openai` or `groq`.
- `DECISION`: human review answer at the end of the analysis. `proceed` runs the debate, `end` goes straight to the memo.

In [ ]:
import getpass
import os
import site
from pathlib import Path

LLM_PROVIDER = "openai"
REASONING_LEVEL = "high"

ROOT = Path("/content/Due-Diligence-MAS")
SRC = ROOT / "src"

SETTINGS = {
    "DEFAULT_LLM_PROVIDER": LLM_PROVIDER,
    "DEFAULT_REASONING_LEVEL": REASONING_LEVEL,
    "AGENT_VERBOSE": "false",
    "CREW_VERBOSE": "false",
    "FLOW_VERBOSE": "false",
    "CREWAI_TRACING_ENABLED": "false",
    "LANGSMITH_TRACING": "false",
    "SKIP_DEBATE_FORMATTER": "true",
    "ANALYST_ASYNC": "true",
    "EXECUTIONS_DB": str(ROOT / "database" / "executions.db"),
    "CREWAI_FLOW_DB_PATH": str(ROOT / "due_diligence_crewai.db"),
    "OTEL_SDK_DISABLED": "true",
}
os.environ.update(SETTINGS)


def load_secret(name, required=True):
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    if not value and required:
        value = getpass.getpass(f"{name}: ")
    if value:
        os.environ[name] = value
    return value


load_secret("OPENAI_API_KEY" if LLM_PROVIDER == "openai" else "GROQ_API_KEY")
load_secret("TAVILY_API_KEY")
load_secret("GITHUB_TOKEN", required=False)
if load_secret("LANGSMITH_API_KEY", required=False):
    os.environ.update({"LANGSMITH_TRACING": "true", "LANGSMITH_PROJECT": "due_diligence"})

SRC.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
(Path(site.getsitepackages()[0]) / "due_diligence_mas.pth").write_text(str(SRC) + "\n")
print(f"provider={LLM_PROVIDER} root={ROOT}")

The `.pth` file above makes the package importable in subprocesses too. The MCP stdio subprocess does not inherit `PYTHONPATH`, so this is required.

## 2. Utils and shared functions

Code used by both frameworks. Each cell writes one file of the package `due_diligence_mas` (same code as the repo). Nothing runs here.

### Package skeleton

In [ ]:
from pathlib import Path

PKG = Path("/content/Due-Diligence-MAS/src/due_diligence_mas")
for rel in [".", "api_gateway", "mas_definition/agents", "mas_definition/agents/adapters", "mas_definition/agents/analysts", "mas_definition/agents/debate", "mas_definition/agents/writer", "mas_definition/classes", "mas_definition/constants", "mas_definition/crewai_fr/crews", "mas_definition/crewai_fr/flows", "mas_definition/crewai_fr/utils", "mas_definition/langgraph/classes", "mas_definition/langgraph/graph", "mas_definition/langgraph/node", "mas_definition/providers", "mas_definition/tools", "mas_definition/utils", "mcp", "utils"]:
    (PKG / rel).mkdir(parents=True, exist_ok=True)
for rel in ["__init__.py", "mas_definition/agents/adapters/__init__.py", "mas_definition/constants/__init__.py", "mas_definition/crewai_fr/__init__.py", "mas_definition/utils/__init__.py"]:
    (PKG / rel).touch()
(PKG / "__init__.py").touch()

### Logging and session id

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/utils/logger.py
import logging
import threading
from pathlib import Path
from typing import Literal

LOG_DIR = Path("log")

LogType = Literal["lg_api_gateway", "ca_api_gateway", "crew", "node", "mcp_task"]


def setup_logger(log_type: LogType, enable_console=False) -> logging.Logger:
    """Create a logger writing to log/<log_type>/{info,error}.log."""
    logger = logging.getLogger(f"{log_type}_logger")
    if logger.handlers:
        return logger

    log_dir = LOG_DIR / log_type
    log_dir.mkdir(parents=True, exist_ok=True)
    logger.setLevel(logging.DEBUG)

    error_file_handler = logging.FileHandler(log_dir / "error.log")
    error_file_handler.setLevel(logging.ERROR)

    info_file_handler = logging.FileHandler(log_dir / "info.log")
    info_file_handler.setLevel(logging.DEBUG)

    formator = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    error_file_handler.setFormatter(formator)
    info_file_handler.setFormatter(formator)

    logger.addHandler(error_file_handler)
    logger.addHandler(info_file_handler)

    if enable_console:
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter('\n %(asctime)s - %(levelname)s - %(message)s \n'))
        logger.addHandler(console_handler)

    return logger


def _log_event(log_type: LogType, source: str, status: str, details: dict):
    extra = " | ".join(f"{k}={v}" for k, v in details.items())
    setup_logger(log_type).info(f"[{source}] [thread={threading.current_thread().name}] {status}" + (f" | {extra}" if extra else ""))


def log_node(node: str, status: str, **details):
    """Log a LangGraph node status line with its thread to log/node, to check that the fan-out branches run in parallel."""
    _log_event("node", node, status, details)


def log_crew(crew: str, status: str, **details):
    """Log a CrewAI crew or flow step status line to log/crew."""
    _log_event("crew", crew, status, details)


def log_mcp_task(source: str, status: str, **details):
    """Log a task manager MCP server status line to log/mcp_task."""
    _log_event("mcp_task", source, status, details)

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/utils/session_manager.py
import uuid

def create_session_id() -> str:
    return str(uuid.uuid4())

### Constants

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/common.py
from enum import Enum

class Framework(Enum):
    CREW_AI = "crewai"
    LANG_GRAPH = "langgraph"

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/pricing.py
MODEL_PRICING = {
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "openai/gpt-oss-20b": {"input": 0.075 / 1_000_000, "output": 0.30 / 1_000_000},
    "openai/gpt-oss-120b": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/research.py
RESEARCH_RIGOR = [
    "Retrieve data thoroughly, but a missing item never blocks you: for each item try up to 3 different approaches (other queries, the official site and its sub-pages, other sources or tools), then move on.",
    "Never repeat a search that already failed and never invent data.",
    "An item still not found goes in 'Missing information' as 'Not found after search: <sources tried>' ('Access required' if the document is gated); the report is always completed.",
]

ANALYST_REPORT_FORMAT = (
    "A markdown report with: a short summary; the findings as a list, each with its source URL, access date and status "
    "(verified, company claim or to be confirmed); and a 'Missing information' list."
)

CONTEXT_FIRST = "Start from the provided context (URLs, findings, missing items) and search only what is absent or missing there."

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/review.py
ANALYSIS_REVIEW_CRITERIA = (
    "Review the aggregated analysis below, built from the analysts' reports, as the completeness gate before the debate. Give a completeness score from 0 to 10:\n"
    "- 0 = nothing usable, 10 = every area fully covered and source-backed;\n"
    "- check the analysis covers all the topics of each area (no skipped topics), is grounded in the profile facts and cites sources;\n"
    "- check that gaps and open questions are declared explicitly, not hidden;\n"
    "- penalize generic, unsupported or off-target statements;\n"
    "- give ONE overall score for the whole analysis, not one per analyst, and list the concrete missing items."
)

TASK_CREATION_INSTRUCTIONS = (
    "Decide autonomously whether follow-up work is needed:\n"
    "- create a task only for concrete missing items that a team member must resolve; do not create tasks if the analysis is complete;\n"
    "- create AT MOST ONE task per topic (tech, commercial), so at most 2 tasks in total: group all the missing items of a topic in that single task; team gaps are not a topic, mention them in the review only;\n"
    "- call get_users first and assign each task to the user whose role matches its topic;\n"
    "- priority is the highest of the grouped items: HIGH for gaps that block the decision, MEDIUM for important ones, LOW for minor ones;\n"
    "- the description must name the company, the topic and list every missing item of the task.\n"
    "When done, return the review."
)

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/agents_definition.py
"""Agent definitions. `description` feeds the backstory; `goal` is passed separately to the agent."""
from enum import Enum

from due_diligence_mas.mas_definition.constants.research import RESEARCH_RIGOR

class AgentName(Enum):
    
        TECH_REVIEWER = "Tech Reviewer"
        COMMERCIAL_ANALYST = "Commercial Analyst"
        COMPANY_PROFILER = "Company Profiler"
        ANALYST_SUPERVISOR = "Analyst Supervisor"
        TEAM_ANALYST = "Team Analyst"
        
        DEBATE_FORMATTER = "Debate Formatter"
        OPPORTUNITY_ANALYST = "Opportunity Analyst"
        RISK_ANALYST = "Risk Analyst"
        DEBATE_JUDGE = "Debate Judge"
        
        MEMO_WRITER = "Memo Writer"

AGENTS = {
    "Tech Reviewer": {
        "name": "Tech Reviewer",
        "role": (
            "Product and technology diligence specialist focused on product "
            "maturity, technical defensibility, scalability, and technology risks."
        ),
        "description": (
            "You assess startups from a product and technology standpoint. "
            "You separate verifiable technical evidence from marketing "
            "language and you flag risks such as replicability, dependencies "
            "and unsupported AI or IP claims."
        ),
        "goal": (
            """ 
                Analyze the company's products and services using its official website,
                product pages, documentation, pricing, FAQs, and customer case studies,
                and cross-check them against public reviews (g2.com, capterra.com, marketplaces).
                Distinguish marketing claims from documented capabilities,
                and flag missing or conflicting information.
            """
        ),
        "responsibilities": [
            "product functionality",
            "product maturity",
            "technical complexity",
            "scalability",
            "integrations",
            "AI/data/IP claims, if applicable",
            "security and compliance claims",
            "technical dependencies",
            "API or developer documentation",
            "open-source or GitHub signals, if available",
            "replicability risk",
            "technology-related red flags",
        ],
        "constraints": [
            "Avoid assuming technical quality without evidence.",
            (
                "If AI, automation, proprietary models, or unique data are "
                "claimed, verify whether the claim is supported or merely "
                "marketing language."
            ),
        ] + RESEARCH_RIGOR,
        "limits": {"max_iter": 8, "max_execution_time": 600},
    },
    "Commercial Analyst": {
        "name": "Commercial Analyst",
        "role": (
            "Market, business model, traction, and financial-signal analyst "
            "focused on evaluating commercial potential."
        ),
        "description": (
            "You assess the commercial side of startups: market, "
            "competition, pricing, go-to-market and traction. You treat "
            "public logos, testimonials and website claims as signals "
            "rather than proof, and you always call out missing financial data. "
            "For funding and revenue news you use the funding news search tool."
        ),
        "goal": (
            "Assess the startup’s commercial attractiveness by analyzing "
            "market context, customer segments, competitors, monetization "
            "model, go-to-market signals, traction indicators, and missing "
            "financial evidence, and compare the competitors' offer with the company's."
        ),
        "responsibilities": [
            "market category",
            "market timing",
            "relevant trends",
            "customer segments",
            "direct and indirect competitors",
            "pricing signals, if public",
            "revenue model",
            "go-to-market model",
            "sales motion",
            "visible commercial traction",
            "partnerships or customer references",
            "funding or financial information, if public",
            "key commercial risks",
            "missing financial or traction data",
        ],
        "constraints": [
            "Do not overstate traction.",
            (
                "Public logos, testimonials, or website claims are signals, "
                "not proof of revenue."
            ),
        ] + RESEARCH_RIGOR,
        "limits": {"max_iter": 12, "max_execution_time": 900},
    },
    "Company Profiler": {
        "name": "Company Profiler",
        "role": (
            "Company understanding specialist focused on extracting what the "
            "startup does, who it serves, and how it positions itself."
        ),
        "description": (
            "You are the first analyst to look at a startup. You read the "
            "company's public presence, mainly its website, and produce "
            "the factual baseline (what it does, for whom, how it "
            "positions itself) that every other specialist builds on."
            "You are aware of best practices of company profiling, includind websites useful to get informations."
        ),
        "goal": (
            "All search must be consider current dateTime and location use always available tools to fetch that informations"
            "Build a clear and factual company profile based on available public "
            "information, mainly the company website. "
            "Extract all possible informations and report all of them: never summarize or drop collected facts."
        ),
        "responsibilities": [
            "company description",
            "product or service offering",
            "value proposition",
            "target customers",
            "use cases",
            "positioning",
            "public claims",
            "official GitHub organization and social/LinkedIn URLs",
            "visible clients, partners, testimonials, or case studies",
            "geography or industry focus, if available",
            "Cite sources and dates, cross-check material claims",
            "names and roles of visible team members, with the team page URL for team analysis"
        ],
        "constraints": [
            "Focus on observable and source-backed information.",
            (
                "Avoid interpreting commercial viability, investment "
                "attractiveness, or technical defensibility unless directly evident."
            ),
            "Separate facts from assumptions.",
            "If information is missing, explicitly mark it as missing.",
        ],
        "limits": {"max_iter": 6, "max_execution_time": 420},
    },
    "Analyst Supervisor": {
        "name": "Analyst Supervisor",
        "role": "Coordinates the analysis and guarantees the quality and completeness of the company profile.",
        "description": (
            "You delegate each task to the specialist named in it and you are the quality gate of the "
            "company profile, the baseline every other specialist builds on. You are also the custodian of "
            "the information gathered: every fact returned by any delegation, the earliest ones included, "
            "stays in your final answer. A later delegation completes the earlier results, it never replaces them."
        ),
        "goal": (
            "Get a complete, source-backed company profile and one report from each specialist, "
            "cumulating the output of every delegation without losing any previously obtained information."
        ),
        "responsibilities": [
            "delegate each task once to the specialist named in it, passing the task text unchanged",
            "keep a running record of everything the coworkers return: after each delegation, "
            "add the new facts to the record and never overwrite, shorten or drop facts already in it",
            "build the final answer from the whole record, every entry verbatim with its source, access date and status, "
            "and not from the last delegation alone",
            "check the company profile: confirmed official website, what the company does, sector, "
            "headquarters and founding year, customers or market, GitHub/LinkedIn/team page URLs if they exist, "
            "and a source for every fact",
            "if the profile lacks any of these items, delegate to the profiling coworker once more, "
            "asking only for the missing items and listing what is already found; then proceed anyway",
            "keep track of every topic already requested: a topic requested without success is marked as "
            "'missing (not found after search)' and is never requested again, "
            "except in the single follow-up request, where it may be retried at most once",
            "add to the profile a 'Gaps' list of what is still missing for each specialist",
            "if a specialist report is incomplete, record the gaps as missing instead of delegating again",
            "resolve conflicting findings: primary source over secondary, more recent over older, "
            "verified over claimed over to-be-confirmed; if undecidable, keep both and mark as to be confirmed, with a rationale",
        ],
        "constraints": [
            "Never lose information obtained in a previous delegation: the final answer must contain it in full.",
            "Never summarize, merge away or rephrase the coworkers' facts; only remove exact duplicates.",
            "Never perform research or add facts that the coworkers did not provide.",
            "Never invent information: list it as missing.",
            "Delegate or ask to coworkers at most twice in total (initial request plus one follow-up).",
            "Never delegate a topic that was already requested and not found, unless it is the single follow-up; "
            "after the follow-up, whatever is still not found stays marked as missing.",
        ],
        "limits": {"max_iter": 8, "max_execution_time": 1500},
    },
    "Team Analyst": {
        "name": "Team Analyst",
        "role": (
            "Founder and organization evaluator focused on team quality, "
            "founder-market fit, execution ability, and organizational gaps."
        ),
        "description": (
            "You evaluate the people behind the startup: founders, key hires "
            "and organizational structure. You rely on public, verifiable "
            "profiles and track records, and you highlight gaps between the "
            "team and the company's strategy."
        ),
        "goal": (
            "Assess whether the founding team and visible organization appear "
            "capable of executing the company’s strategy, based on public "
            "information about the team members."
        ),
        "responsibilities": [
            "founder backgrounds",
            "prior companies or exits",
            "domain expertise",
            "founder-market fit",
            "technical and commercial balance",
            "visible leadership team",
            "advisors or investors",
            "hiring signals",
            "organizational gaps",
            "key person risk",
        ],
        "constraints": [
            (
                "Do not overstate quality based only on prestigious names or "
                "previous employers."
            ),
            "Assessment should be evidence-based and clearly mark unknowns.",
        ] + RESEARCH_RIGOR,
        "limits": {"max_iter": 8, "max_execution_time": 600},
    },
    "Debate Formatter": {
        "name": "Debate Formatter",
        "role": (
            "Debate preparation agent that converts specialist outputs into a "
            "concise, structured, evidence-based debate pack."
        ),
        "description": (
            "You prepare the debate phase. You condense the specialists' "
            "outputs into a structured pack in which every claim is "
            "traceable to its source, so that the debating agents never "
            "need the raw analyses."
        ),
        "goal": (
            "Prepare a clean and traceable evidence base that PRO, CONTRO, and "
            "Judge agents can use without re-reading all raw analysis."
        ),
        "responsibilities": [
            "summarize the company snapshot",
            "extract the most relevant evidence items",
            "remove duplication",
            "separate facts, assumptions, and missing data",
            "assign IDs to evidence items",
            "identify initial tensions",
            "preserve confidence levels",
            "ensure all debate arguments can reference evidence IDs",
        ],
        "constraints": [
            "Must not create new facts or add new analysis.",
        ],
        "limits": {"max_iter": 2, "max_execution_time": 120},
    },
    "Opportunity Analyst": {
        "name": "Opportunity Analyst",
        "role": (
            "Positive investment case extractor focused on building the "
            "strongest evidence-backed argument in favor of the startup."
        ),
        "description": (
            "You argue the PRO side of the investment case. You work "
            "only from the formatted evidence pack and build the "
            "strongest argument that is actually supported by "
            "evidence, without inventing facts."
        ),
        "goal": (
            "Identify the strongest reasons to continue diligence or consider "
            "investment, using only the formatted debate evidence pack."
        ),
        "responsibilities": [
            "strongest positive signals",
            "attractive market or timing signals",
            "commercial upside",
            "product strengths",
            "technical defensibility",
            "founder-market fit",
            "visible traction signals",
            "reasons to continue diligence",
        ],
        "constraints": [
            "Must link every argument to evidence IDs.",
            "Cannot invent new facts.",
            "Cannot search for external evidence.",
            "Cannot ignore missing data.",
            "If a positive argument is based on weak evidence, state that clearly.",
        ],
        "limits": {"max_iter": 2, "max_execution_time": 120},
    },
    "Risk Analyst": {
        "name": "Risk Analyst",
        "role": (
            "Negative investment case extractor focused on challenging the PRO case "
            "and identifying risks, weaknesses, and deal breakers."
        ),
        "description": (
            "You argue the CONTRO side of the investment case. You work from "
            "the formatted evidence pack, challenge the PRO arguments and "
            "surface weaknesses, risks and deal breakers backed by evidence."
        ),
        "goal": (
            "Build the strongest evidence-backed case against the investment, "
            "directly addressing the PRO arguments where relevant."
        ),
        "responsibilities": [
            "weak or unsupported PRO arguments",
            "missing critical evidence",
            "commercial risks",
            "market risks",
            "product or technical risks",
            "team risks",
            "financial uncertainty",
            "possible deal breakers",
            "areas where public signals are insufficient",
        ],
        "constraints": [
            "Must not simply be pessimistic.",
            (
                "Every negative argument should reference evidence item IDs or "
                "missing data from the debate pack."
            ),
        ],
        "limits": {"max_iter": 2, "max_execution_time": 120},
    },
    "Debate Judge": {
        "name": "Debate Judge",
        "role": (
            "Adversarial debate evaluator responsible for weighing PRO and CONTRO "
            "arguments and producing a balanced investment judgement."
        ),
        "description": (
            "You are the impartial judge of the PRO/CONTRO debate. You weigh "
            "the quality of the evidence on both sides, identify the central "
            "investment tension and issue a balanced preliminary judgement."
        ),
        "goal": (
            "Compare the positive and negative cases, assess evidence quality, "
            "identify the central investment tension, and produce a preliminary "
            "judgement for the memo."
        ),
        "responsibilities": [
            "strength of PRO arguments",
            "severity of CONTRO arguments",
            "quality of supporting evidence",
            "unsupported or weak claims",
            "critical missing data",
            "whether risks are knowable, mitigable, or deal-breaking",
            "the central tension of the investment case",
            "whether further diligence is justified",
        ],
        "constraints": [
            "Must not write the final memo.",
            "Produces the debate conclusion that the final memo will later use.",
        ],
        "limits": {"max_iter": 2, "max_execution_time": 120},
    },
    "Memo Writer": {
        "name": "Memo Writer",
        "role": (
            "Final due diligence memo composer responsible for transforming validated "
            "agent outputs into a structured investment memo."
        ),
        "description": (
            "You write the final due diligence memo. You turn the validated "
            "outputs of the specialists and the debate judge into a "
            "structured, professional document, without adding information "
            "that is not in those outputs."
        ),
        "goal": (
            "Produce a clear, professional, evidence-based due diligence memo using "
            "the outputs from the specialist agents and the debate judge."
        ),
        "responsibilities": [
            "executive summary",
            "company overview",
            "commercial analysis",
            "product and technology review",
            "team assessment",
            "debate summary",
            "investment judgement",
            "key risks",
            "open questions",
            "recommended next diligence steps",
        ],
        "constraints": [
            "Does not perform new research.",
            "Does not add new claims.",
            "Never omits or generalizes findings from the analysis: completeness over brevity.",
            "Summarizes the debate; reports the analysis in full.",
            "Does not override the Judge’s conclusion.",
            "Must clearly separate facts, assumptions, risks, and missing data.",
            "If evidence is weak, the memo should say so explicitly.",
        ],
        "limits": {"max_iter": 2, "max_execution_time": 300},
    },
}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/tools_definition.py
"""Tool catalogue. `implemented` marks tools actually wired to agents; the others are planned."""

TOOLS = [{'name': 'Crawler (Tavily based)',
  'implemented': False,
  'description': 'Map the site, then crawl relevant pages and return content by page type',
  'agents': ['Company Profiler', 'Tech Reviewer'],
  'input': 'URL',
  'output': {'type': 'list_of_pages',
             'fields': ['URL', 'page type', 'markdown content', 'access date'],
             'page_types': ['home',
                            'about',
                            'team',
                            'pricing',
                            'product',
                            'case studies',
                            'blog',
                            'careers',
                            'docs',
                            'security/legal']}},
 {'name': 'Web search (Tavily)',
  'implemented': True,
  'description': 'Search the web and news (Commercial Analyst also has a funding-news search restricted to financial sources)',
  'agents': ['Company Profiler', 'Commercial Analyst'],
  'input': {'query': 'string', 'topic': 'general/news', 'max_results': 'number'},
  'output': {'type': 'list_of_results', 'fields': ['title', 'URL', 'snippet', 'publication date']}},
 {'name': 'Page extract (Tavily Extract)',
  'implemented': True,
  'description': 'Read specific known pages without crawling',
  'agents': ['Company Profiler', 'Commercial Analyst', 'Tech Reviewer', 'Team Analyst'],
  'input': {'urls': 'list of URLs', 'max_urls': 20},
  'output': {'type': 'per_url_result',
             'fields': ['markdown content', 'access date', 'error if unreachable']}},
 {'name': 'GitHub',
  'implemented': True,
  'description': 'Search public repositories by organization, author or title',
  'agents': ['Tech Reviewer'],
  'input': {'author_or_org_or_repo_url': 'string', 'query': 'string'},
  'output': {'type': 'repositories',
             'fields': ['name',
                        'URL',
                        'description',
                        'stars',
                        'contributors',
                        'last commit date',
                        'license',
                        'languages']}},
 {'name': 'People search (Exa)',
  'implemented': False,
  'description': 'Search public profiles of founders and team members, category = people',
  'agents': ['Team Analyst'],
  'input': {'name': 'string', 'company': 'string', 'role': 'string'},
  'output': {'type': 'profiles',
             'fields': ['name', 'role', 'company', 'location', 'URL', 'highlights']}},
 {'name': 'Company / competitor search (Exa)',
  'implemented': False,
  'description': 'Find similar companies and competitors, category = company + findSimilar',
  'agents': ['Commercial Analyst'],
  'input': {'url_or_company_description': 'string'},
  'output': {'type': 'companies', 'fields': ['name', 'URL', 'description', 'highlights']}},
 {'name': 'Company registry / funding lookup',
  'implemented': False,
  'description': 'Look up corporate data, funding rounds and previous exits',
  'agents': ['Commercial Analyst', 'Team Analyst'],
  'input': {'company_name': 'string', 'country': 'string'},
  'output': {'fields': ['legal name',
                        'incorporation date',
                        'known rounds',
                        'investors',
                        'source']}},
 {'name': 'Package registry lookup',
  'implemented': False,
  'description': 'Verify published SDKs or models across npm, PyPI, Hugging Face',
  'agents': ['Tech Reviewer'],
  'input': {'package_name_or_organization': 'string'},
  'output': {'type': 'packages',
             'fields': ['name', 'version', 'downloads', 'last release', 'license', 'URL']}},
 {'name': 'Review & traction signals',
  'implemented': False,
  'description': 'Collect reviews from G2, Trustpilot, app stores, Product Hunt launches and job '
                 'postings',
  'agents': ['Commercial Analyst', 'Tech Reviewer'],
  'input': {'product_or_company_name': 'string'},
  'output': {'type': 'signals',
             'fields': ['source', 'review count and score', 'launch', 'open positions', 'URL']}},
 {'name': 'Patent search',
  'implemented': False,
  'description': 'Search patents, only when IP claims are present',
  'agents': ['Tech Reviewer'],
  'input': {'assignee': 'string', 'keywords': 'string'},
  'output': {'type': 'patents', 'fields': ['number', 'title', 'date', 'status', 'URL']}},
 {'name': 'Current date/time',
  'implemented': False,
  'description': 'Return the current date and time to assess source freshness',
  'agents': ['All specialists'],
  'input': None,
  'output': {'type': 'ISO timestamp'}},
 {'name': 'Schema validator',
  'implemented': False,
  'description': "Superseded by Pydantic validation of SpecialistReport (a manager agent cannot hold tools)",
  'agents': ['Analyst Supervisor'],
  'input': {'agent_json_output': 'object'},
  'output': {'fields': ['valid yes/no', 'list of missing or non-compliant fields']}}]

### Data classes

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/classes/company.py
from typing import Optional
from pydantic import BaseModel, Field

class CompanyInfo(BaseModel):
    """Target company information provided by the user."""

    company_name: Optional[str] = None
    company_site: Optional[str] = None
    company_sector: Optional[str] = None
    additional_info: Optional[str] = None

    def describe(self) -> str:
        parts = [self.company_name or "Unknown company"]
        if self.company_site:
            parts.append(f"official website: {self.company_site}")
        if self.company_sector:
            parts.append(f"sector: {self.company_sector}")
        if self.additional_info:
            parts.append(f"additional information: {self.additional_info}")
        return ", ".join(parts)


class CompanyProfile(BaseModel):
    """Profiling output: shared facts plus one list of entries per specialist."""

    generic: Optional[list[str]] = Field(default=None, description="Generic company info, one entry per fact: fact | source URL | access date | status")
    tech: Optional[list[str]] = Field(default=None, description="Inputs and gaps for technology review, one entry per fact or gap")
    commercial: Optional[list[str]] = Field(default=None, description="Inputs and gaps for commercial analysis, one entry per fact or gap")
    team: Optional[list[str]] = Field(default=None, description="Inputs and gaps for team analysis, one entry per fact or gap")

    def for_specialist(self, specialist: str) -> str:
        """Generic info plus the entries of the given specialist: tech, commercial or team."""
        generic = "\n".join(f"- {e}" for e in self.generic or [])
        own = "\n".join(f"- {e}" for e in getattr(self, specialist) or [])
        return f"GENERIC INFO:\n{generic}\n\n{specialist.upper()} INPUTS:\n{own}"


class ProfileReview(BaseModel):
    """Supervisor verdict on the company profile."""

    approved: bool = Field(description="True only if the profile is complete, source-backed and usable by the specialists")
    issues: list[str] = Field(default_factory=list, description="Concrete problems found: missing fields, facts without source URL, unsupported claims, gaps not declared")
    feedback: str = Field(default="", description="Actionable instructions for the profiler to fix the issues; empty if approved")


class AnalysisReview(BaseModel):
    """Supervisor completeness verdict on the analysts' results."""

    score: int = Field(ge=0, le=10, description="Single overall completeness score of all the analyses together, from 0 (nothing usable) to 10 (fully complete)")
    missing: list[str] = Field(description="Concrete missing topics, unsupported claims or undeclared gaps; empty if none")
    summary: str = Field(description="Short motivation of the score")


class ProfiledCompany(BaseModel):
    """Target company info together with the profile produced by the profiling crew."""

    company_info: Optional[CompanyInfo] = None
    company_profile: Optional[CompanyProfile] = None

    def describe(self) -> str:
        return self.company_info.describe()

    def profile_topic(self) -> str:
        """Company description plus the generic info from the profiling crew."""
        generic = "\n".join(f"- {e}" for e in self.company_profile.generic or []) if self.company_profile else ""
        return f"{self.describe()}\n\nCOMPANY PROFILE:\n{generic}"

    def topic_for(self, specialist: str) -> str:
        """Company description plus generic info and the entries of the given specialist: tech, commercial or team."""
        if not self.company_profile:
            return self.describe()
        return f"{self.describe()}\n\n{self.company_profile.for_specialist(specialist)}"

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/classes/context.py
from pydantic import BaseModel


class Location(BaseModel):
    city: str
    timezone: str
    country: str
    region: str


class CurrentContext(BaseModel):
    location: Location | str
    date_time: str

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/classes/__init__.py
from due_diligence_mas.mas_definition.classes.company import CompanyInfo, ProfiledCompany, CompanyProfile
from due_diligence_mas.mas_definition.classes.context import CurrentContext, Location

__all__ = ["CompanyInfo", "ProfiledCompany", "CompanyProfile", "CurrentContext", "Location"]

### LLM provider

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/providers/llm_provider.py
import os
from crewai import LLM
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


class GroqSafeLLM(LLM):
    """Strips crewai's `cache_breakpoint` flag, which Groq rejects via LiteLLM."""

    def _format_messages_for_provider(self, messages):
        cleaned = [
            {k: v for k, v in m.items() if k != "cache_breakpoint"}
            if isinstance(m, dict) else m
            for m in messages or []
        ]
        return super()._format_messages_for_provider(cleaned)


class LLM_PROVIDER():
    
    def __init__(self, provider="", reasoning="", temperature=0.2):
        
        load_dotenv()
        
        if reasoning == "": 
            reasoning = os.getenv("DEFAULT_REASONING_LEVEL", "medium")
            
        if provider == "":
            provider = os.getenv("DEFAULT_LLM_PROVIDER", "groq")

        self.temperature = temperature
        
        match provider:
            case "groq":
                
                if os.getenv("GROQ_API_KEY") is None:
                    raise ValueError("GROQ_API_KEY environment variable is not set.")
                
                if reasoning == "medium":
                    self.model = "groq/openai/gpt-oss-20b"
                elif reasoning == "high":
                    self.model = "groq/openai/gpt-oss-120b"
                else:
                    raise ValueError("Invalid reasoning level. Choose 'medium' or 'high'.")
                
                self.llm = GroqSafeLLM(
                                    model=self.model,
                                    temperature=self.temperature,
                                )
              
            case "openai":
                if os.getenv("OPENAI_API_KEY") is None:
                    raise ValueError("OPENAI_API_KEY environment variable is not set.")
                
                if reasoning == "medium":
                    self.model = "openai/gpt-4o-mini"
                elif reasoning == "high":
                    self.model = "openai/gpt-4o-mini"
                else:
                    raise ValueError("Invalid reasoning level. Choose 'medium' or 'high'.")
               
                self.llm = LLM(
                        model=self.model,
                        temperature=self.temperature
                    )
        

    def get_llm(self):
        return self.llm 

    def get_chat_model(self):

        model_provider, model_name = self.model.split("/", 1)
        return init_chat_model(model=model_name, model_provider=model_provider, temperature=self.temperature)
         

### Utilities: names, backstory, execution DB, cost and usage tracking

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/agent_names.py
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


def agent_slug(agent_name: AgentName | str) -> str:
    """Snake_case identifier of an agent, e.g. "Tech Reviewer" -> "tech_reviewer"."""
    value = agent_name.value if isinstance(agent_name, AgentName) else agent_name
    return value.lower().replace(" ", "_")


def get_node_name(name: str) -> str:
    """Name of a LangGraph node, always "<snake_case_name>_node", e.g. "request handler" -> "request_handler_node"."""
    slug = name.lower().replace(" ", "_")
    return slug if slug.endswith("_node") else f"{slug}_node"


def get_agent_node_name(agent_name: AgentName | str) -> str:
    """Name of the LangGraph node that runs the agent, e.g. "Tech Reviewer" -> "tech_reviewer_node"."""
    return get_node_name(agent_slug(agent_name))

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/backstory_builder.py
from due_diligence_mas.mas_definition.constants.agents_definition import AGENTS


def backstory_builder(agent_name: str) -> str:
    """
    Builds a backstory for the given agent name.

    Args:
        agent_name (str): The name of the agent.

    Returns:
        str: A backstory for the agent.

    Raises:
        KeyError: If the agent is not defined in agents_definition.
    """
    if agent_name not in AGENTS:
        raise KeyError(f"Unknown agent '{agent_name}'. Available: {', '.join(AGENTS)}")
    agent = AGENTS[agent_name]

    parts = [f"You are the {agent['name']}.", agent["description"]]
    if agent.get("role"):
        parts.append(f"Role: {agent['role']}")
    if agent.get("skills"):
        parts.append("Skills:\n" + "\n".join(f"- {s}" for s in agent["skills"]))
    if agent.get("responsibilities"):
        parts.append("Responsibilities:\n" + "\n".join(f"- {r}" for r in agent["responsibilities"]))
    if agent.get("constraints"):
        parts.append("Constraints:\n" + "\n".join(f"- {c}" for c in agent["constraints"]))
    return "\n\n".join(parts)
    
    

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/execution_db.py
"""
Local SQLite db of the executions (same file also holds the LangGraph checkpoints).

Tables:
    executions: one row per execution (thread), updated while the graph runs.
    trackers:   usage totals (LLM calls, tokens, time) kept while an execution waits for the human review.
"""
import json
import os
import sqlite3
from contextlib import closing
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Optional

DEFAULT_EXECUTIONS_DB = Path(__file__).resolve().parents[4] / "database" / "executions.db"


class ExecutionStatus(str, Enum):
    RUNNING = "running"
    AWAITING_REVIEW = "awaiting_review"
    COMPLETED = "completed"
    FAILED = "failed"


EXECUTIONS_SCHEMA = """
CREATE TABLE IF NOT EXISTS executions (
    thread_id         TEXT PRIMARY KEY,  -- session id, same as the LangGraph thread id
    company_name      TEXT,
    framework         TEXT NOT NULL,     -- langgraph / crewai
    status            TEXT NOT NULL,     -- ExecutionStatus
    current_node      TEXT,              -- last node finished (human_review while paused), NULL when completed
    current_crew      TEXT,              -- CrewAI only, NULL for LangGraph
    metrics           TEXT,              -- JSON: time, LLM calls, tokens, tracing
    memo_markdown     TEXT,              -- final memo
    pending_review    TEXT,              -- JSON: analysis preview and options shown to the human while paused
    created_at        TEXT NOT NULL,
    updated_at        TEXT NOT NULL
)
"""

TRACKERS_SCHEMA = """
CREATE TABLE IF NOT EXISTS trackers (
    thread_id TEXT PRIMARY KEY,
    snapshot  TEXT NOT NULL              -- JSON from UsageTracker.snapshot()
)
"""

JSON_COLUMNS = ("metrics", "pending_review")
UPDATABLE_COLUMNS = {
    "company_name", "status", "current_node", "current_crew", "metrics", "memo_markdown", "pending_review",
}

LEGACY_COLUMNS = {"company": "company_name", "memo": "memo_markdown", "interrupt": "pending_review"}


def db_path() -> Path:
    path = Path(os.getenv("EXECUTIONS_DB") or DEFAULT_EXECUTIONS_DB)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _connect() -> sqlite3.Connection:
    connection = sqlite3.connect(db_path())
    connection.row_factory = sqlite3.Row
    connection.execute(EXECUTIONS_SCHEMA)
    connection.execute(TRACKERS_SCHEMA)
    _migrate_legacy_columns(connection)
    return connection


def _migrate_legacy_columns(connection: sqlite3.Connection) -> None:
    existing = {row["name"] for row in connection.execute("PRAGMA table_info(executions)")}
    for old, new in LEGACY_COLUMNS.items():
        if old in existing:
            connection.execute(f"ALTER TABLE executions RENAME COLUMN {old} TO {new}")
    if "pending_review" not in existing and "interrupt" not in existing:
        connection.execute("ALTER TABLE executions ADD COLUMN pending_review TEXT")
    connection.execute("UPDATE executions SET status = 'awaiting_review' WHERE status = 'interrupted'")
    connection.commit()


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def create_execution(thread_id: str, framework: str, company_name: Optional[str] = None) -> None:
    """Insert the execution as running; a thread that already exists is restarted."""
    now = _now()
    with closing(_connect()) as connection, connection:
        connection.execute(
            "INSERT OR REPLACE INTO executions (thread_id, company_name, framework, status, created_at, updated_at) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (thread_id, company_name, framework, ExecutionStatus.RUNNING.value, now, now),
        )


def update_execution(thread_id: str, **fields) -> None:
    """Update the given columns (dicts are stored as JSON, `status` can be an ExecutionStatus) and refresh `updated_at`."""
    unknown = set(fields) - UPDATABLE_COLUMNS
    if unknown:
        raise ValueError(f"Unknown execution columns: {sorted(unknown)}")

    for column in JSON_COLUMNS:
        if isinstance(fields.get(column), dict):
            fields[column] = json.dumps(fields[column], ensure_ascii=False)
    if isinstance(fields.get("status"), ExecutionStatus):
        fields["status"] = fields["status"].value

    assignments = ", ".join(f"{column} = ?" for column in [*fields, "updated_at"])
    with closing(_connect()) as connection, connection:
        connection.execute(
            f"UPDATE executions SET {assignments} WHERE thread_id = ?", (*fields.values(), _now(), thread_id)
        )


def get_execution(thread_id: str) -> Optional[dict]:
    """Return the execution row (JSON columns parsed) or None if the thread does not exist."""
    with closing(_connect()) as connection:
        row = connection.execute("SELECT * FROM executions WHERE thread_id = ?", (thread_id,)).fetchone()
    if row is None:
        return None

    execution = dict(row)
    for column in JSON_COLUMNS:
        if execution[column]:
            execution[column] = json.loads(execution[column])
    return execution


def save_tracker(thread_id: str, snapshot: dict) -> None:
    """Store the usage totals of an execution, so they survive between run() and resume()."""
    with closing(_connect()) as connection, connection:
        connection.execute(
            "INSERT OR REPLACE INTO trackers (thread_id, snapshot) VALUES (?, ?)", (thread_id, json.dumps(snapshot))
        )


def load_tracker(thread_id: str) -> Optional[dict]:
    with closing(_connect()) as connection:
        row = connection.execute("SELECT snapshot FROM trackers WHERE thread_id = ?", (thread_id,)).fetchone()
    return json.loads(row["snapshot"]) if row else None


def delete_tracker(thread_id: str) -> None:
    with closing(_connect()) as connection, connection:
        connection.execute("DELETE FROM trackers WHERE thread_id = ?", (thread_id,))

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/extract_agent.py
from due_diligence_mas.mas_definition.constants.agents_definition import AGENTS

def extract_agent(agent_name: str) -> dict:
    
    """
       Extracts the agent's information from the AGENTS dictionary.
       
        Args:
            agent_name (str): The name of the agent to extract.
                   
        Returns:
            dict: A dictionary containing the agent's information. 
    """
    if agent_name not in AGENTS:
        raise KeyError(f"Unknown agent '{agent_name}'. Available: {', '.join(AGENTS)}")
    
    return AGENTS[agent_name]


def agent_limits(agent_name: str) -> dict:
    """Inference limits (max_iter, max_execution_time) to pass to the Agent constructor."""
    return dict(extract_agent(agent_name).get("limits", {}))

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/llm_cost.py
from typing import Optional

from due_diligence_mas.mas_definition.constants.pricing import MODEL_PRICING


def get_model_pricing(model: str) -> Optional[dict]:
    """Per-token prices of the model (longest matching key of MODEL_PRICING), None if the model is not priced."""
    matches = [key for key in MODEL_PRICING if key in model]
    return MODEL_PRICING[max(matches, key=len)] if matches else None


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> Optional[dict]:
    """
    Cost in USD of the given tokens.

    Returns:
        {"input": ..., "output": ..., "total": ...}, or None if the model has no price in MODEL_PRICING
    """
    pricing = get_model_pricing(model)
    if pricing is None:
        return None

    input_cost = input_tokens * pricing["input"]
    output_cost = output_tokens * pricing["output"]
    return {"input": input_cost, "output": output_cost, "total": input_cost + output_cost}


def summarize_cost(models: dict) -> dict:
    """
    Cost in USD of the tokens by model: total and per model. Models without a price are listed in
    `unpriced_models` and not counted.

    Args:
        models: {model name: {"prompt_tokens": ..., "completion_tokens": ...}}
    """
    by_model, unpriced = {}, []
    for model, totals in models.items():
        cost = calculate_cost(model, totals["prompt_tokens"], totals["completion_tokens"])
        if cost is None:
            unpriced.append(model)
        else:
            by_model[model] = {key: round(value, 6) for key, value in cost.items()}

    return {
        "input": round(sum(cost["input"] for cost in by_model.values()), 6),
        "output": round(sum(cost["output"] for cost in by_model.values()), 6),
        "total": round(sum(cost["total"] for cost in by_model.values()), 6),
        "by_model": by_model,
        "unpriced_models": unpriced,
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/request_handling.py

from due_diligence_mas.mas_definition.classes import CompanyInfo


def print_welcome_message():
    """ 
        Function useful to print basic information of Due diligence MAS
    """
    print("=" * 80)
    print(f"{'Due Diligence Flow':^80}")
    print("\n   Welcome to the Due Diligence Flow. \n   This flow will guide you through the process of analyzing a target company,\n   reviewing its technology, commercial aspects, and team dynamics, and\n   generating a comprehensive report.\n")
    print("=" * 80)
    print("\n")

def request_company_info():
    
    return CompanyInfo(
        company_name= input("Enter the target company name: "),
        company_site= input("Enter the target company website (optional): "),
        company_sector= input("Enter the target company sector (optional): "),
        additional_info= input("Enter additional informations related to the target company (optional): ")  
    )
    
         
     

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/utils/usage_tracker.py
import os
import threading
import time
from contextlib import contextmanager
from typing import Any, Optional
from uuid import UUID

from langchain_core.callbacks import BaseCallbackHandler

from due_diligence_mas.mas_definition.utils.llm_cost import summarize_cost


class UsageTracker(BaseCallbackHandler):
    """
    LangChain callback that totals LLM calls and tokens of a LangGraph run.

    Pass it once in the graph config (`config={"callbacks": [tracker]}`): LangGraph propagates it to every node,
    including parallel ones. It lives in memory, use `snapshot` / `restore` to carry the totals over a resume
    in another process.
    """

    def __init__(self):
        self._lock = threading.Lock()
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.active_s = 0.0
        self.models: dict[str, dict[str, int]] = {}
        self.root_run_ids: list[str] = []

    def on_chain_start(self, serialized: Any, inputs: Any, *, run_id: UUID, parent_run_id: Optional[UUID] = None, **kwargs: Any) -> None:
        if parent_run_id is None:
            with self._lock:
                self.root_run_ids.append(str(run_id))

    def on_llm_end(self, response, **kwargs: Any) -> None:
        usage, model = {}, "unknown"
        try:
            message = response.generations[0][0].message
            usage = message.usage_metadata or {}
            model = message.response_metadata.get("model_name") or model
        except (IndexError, AttributeError):
            pass
        input_tokens, output_tokens = usage.get("input_tokens", 0), usage.get("output_tokens", 0)
        with self._lock:
            self.calls += 1
            self.prompt_tokens += input_tokens
            self.completion_tokens += output_tokens
            totals = self.models.setdefault(model, {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0})
            totals["calls"] += 1
            totals["prompt_tokens"] += input_tokens
            totals["completion_tokens"] += output_tokens

    @contextmanager
    def measure(self):
        """Add the elapsed time of the block to `active_s`; the human wait between two invokes stays out."""
        start = time.perf_counter()
        try:
            yield
        finally:
            with self._lock:
                self.active_s += time.perf_counter() - start

    def snapshot(self) -> dict:
        with self._lock:
            return {
                "calls": self.calls,
                "prompt_tokens": self.prompt_tokens,
                "completion_tokens": self.completion_tokens,
                "active_s": self.active_s,
                "models": {model: dict(totals) for model, totals in self.models.items()},
                "root_run_ids": list(self.root_run_ids),
            }

    @classmethod
    def restore(cls, snapshot: dict) -> "UsageTracker":
        tracker = cls()
        tracker.calls = snapshot.get("calls", 0)
        tracker.prompt_tokens = snapshot.get("prompt_tokens", 0)
        tracker.completion_tokens = snapshot.get("completion_tokens", 0)
        tracker.active_s = snapshot.get("active_s", 0.0)
        tracker.models = {model: dict(totals) for model, totals in snapshot.get("models", {}).items()}
        tracker.root_run_ids = list(snapshot.get("root_run_ids", []))
        return tracker

    def metrics(self) -> dict:
        snap = self.snapshot()
        return {
            "total_time_s": round(snap["active_s"], 2),
            "llm_calls": snap["calls"],
            "tokens": {
                "prompt": snap["prompt_tokens"],
                "completion": snap["completion_tokens"],
                "total": snap["prompt_tokens"] + snap["completion_tokens"],
            },
            "cost_usd": summarize_cost(snap["models"]),
            "tracing": {
                "langsmith_enabled": os.getenv("LANGSMITH_TRACING", "").lower() == "true",
                "langsmith_project": os.getenv("LANGSMITH_PROJECT"),
                "root_run_ids": snap["root_run_ids"],
            },
        }

### Tools: Apify, GitHub, Tavily, request handler

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/tools/apify.py
from dotenv import load_dotenv
from due_diligence_mas.mas_definition.constants.common import Framework

load_dotenv()

APIFY_ACTORS = {
    "profile": "dev_fusion/linkedin-profile-scraper",
}


def _tool_class(framework: Framework):
    """Apify actor tool class of the framework, imported lazily so only the one in use is required."""
    match framework:
        case Framework.CREW_AI:
            from crewai_tools import ApifyActorsTool
            return ApifyActorsTool
        case Framework.LANG_GRAPH:
            from langchain_apify import ApifyActorsTool
            return ApifyActorsTool
        case _:
            raise ValueError(f"Unknown framework '{framework}'")


def get_linkedin_tool(tool_type: str = "profile", framework: Framework = Framework.CREW_AI):
    """LinkedIn tool backed by an Apify actor. tool_type: one of APIFY_ACTORS.
    crewai_tools names the actor argument `actor_name`, langchain_apify `actor_id`."""
    if tool_type not in APIFY_ACTORS:
        raise ValueError(f"Unknown Apify tool type '{tool_type}'. Available: {', '.join(APIFY_ACTORS)}")

    actor = APIFY_ACTORS[tool_type]
    argument = "actor_id" if framework == Framework.LANG_GRAPH else "actor_name"
    return _tool_class(framework)(**{argument: actor})

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/tools/github.py
import os
import re

import requests
from dotenv import load_dotenv

load_dotenv()

API = "https://api.github.com"
TIMEOUT = 15

def _get(path: str, params: dict | None = None) -> requests.Response:
    headers = {"Accept": "application/vnd.github+json"}
    if token := os.getenv("GITHUB_TOKEN"):
        headers["Authorization"] = f"Bearer {token}"
    return requests.get(f"{API}{path}", headers=headers, params=params, timeout=TIMEOUT)


def _contributors(full_name: str) -> int | None:
    """Contributor count of a repo. per_page=1 so the page number in the Link header equals the total."""
    resp = _get(f"/repos/{full_name}/contributors", {"per_page": 1, "anon": "true"})
    if resp.status_code != 200:
        return None
    last = resp.links.get("last", {}).get("url", "")
    match = re.search(r"[?&]page=(\d+)", last)
    return int(match.group(1)) if match else len(resp.json())


def github_repos(owner: str, max_repos: int = 5) -> str:
    """List the top public repositories (by stars) of a GitHub organization or user with
    name, URL, description, stars, forks, license, languages, last push and contributors count.
    `owner` is the organization or user name (e.g. 'openai'); `max_repos` is capped at 10."""
    max_repos = max(1, min(max_repos, 10))
    try:
        resp = _get(f"/users/{owner}/repos", {"per_page": 100, "sort": "pushed"})
        if resp.status_code != 200:
            return f"GitHub data unavailable for '{owner}' (HTTP {resp.status_code})."

        repos = [r for r in resp.json() if not r["fork"] and not r["archived"]]
        repos.sort(key=lambda r: r["stargazers_count"], reverse=True)

        lines = []
        for r in repos[:max_repos]:
            langs = _get(f"/repos/{r['full_name']}/languages").json()
            lines.append(
                f"- {r['name']} ({r['html_url']}): {r.get('description') or 'n/a'} | "
                f"stars={r['stargazers_count']} forks={r['forks_count']} "
                f"license={(r.get('license') or {}).get('spdx_id')} "
                f"languages={list(langs)} last_push={r['pushed_at']} "
                f"contributors={_contributors(r['full_name'])}"
            )
        return "\n".join(lines) or f"No public repositories found for '{owner}'."
    except requests.RequestException as e:
        return f"GitHub request failed: {e}"

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/tools/tavily.py
from dotenv import load_dotenv

from due_diligence_mas.mas_definition.constants.common import Framework
from crewai_tools import TavilyExtractorTool, TavilySearchTool
from langchain_tavily import TavilyExtract, TavilySearch

load_dotenv()

FUNDING_DOMAINS = [
    "finsmes.com",
    "tech.eu",
    "news.crunchbase.com",
    "startupitalia.eu",
    "economyup.it",
]

SEARCH_CONFIGS = {
    "search": {
        "search_depth": "advanced",
        "max_results": 5,
        "include_answer": False,
    },
    "profile_search": {
        "search_depth": "advanced",
        "max_results": 5,
        "include_answer": False,
    },
    "funding_search": {
        "name": "tavily_funding_news_search",
        "description": (
            "Search funding, revenue and financial news about a company, "
            "restricted to startup and financial news sources."
        ),
        "search_depth": "advanced",
        "topic": "news",
        "max_results": 7,
        "include_answer": False,
        "include_domains": FUNDING_DOMAINS,
    },
}


MAX_CONTENT_CHARS = 10000
BLOCKING_ERROR_CODES = ("401", "429", "432", "433")


class TavilyBlockingError(RuntimeError):
    """Tavily refuses every request (invalid key, rate or plan limit): retrying cannot work."""


def _raise_on_blocking_error(output):
    """langchain_tavily returns API errors as {"error": exception} text for the model, which retries until the tool call limit. Account level errors are raised instead; query errors (400) stay with the model."""
    error = output.get("error") if isinstance(output, dict) else None
    if error is not None and any(f"Error {code}" in str(error) for code in BLOCKING_ERROR_CODES):
        raise TavilyBlockingError(str(error))
    return output


def _truncate_results(output, max_chars: int = MAX_CONTENT_CHARS):
    """Cut the text of every result so tool outputs do not fill the agent context."""
    if isinstance(output, dict):
        for item in output.get("results") or []:
            for key in ("content", "raw_content"):
                if isinstance(item.get(key), str) and len(item[key]) > max_chars:
                    item[key] = item[key][:max_chars] + "... [truncated]"
    return output


class _TruncatedTavilySearch(TavilySearch):
    def _run(self, *args, **kwargs):
        return _truncate_results(_raise_on_blocking_error(super()._run(*args, **kwargs)))


class _TruncatedTavilyExtract(TavilyExtract):
    def _run(self, *args, **kwargs):
        return _truncate_results(_raise_on_blocking_error(super()._run(*args, **kwargs)))


def _tool_classes(framework: Framework):
    """(search, extract) tool classes of the framework, imported lazily so only the one in use is required."""
    match framework:
        case Framework.CREW_AI:
            return TavilySearchTool, TavilyExtractorTool
        case Framework.LANG_GRAPH:
            return _TruncatedTavilySearch, _TruncatedTavilyExtract
        case _:
            raise ValueError(f"Unknown framework '{framework}'")


def get_tavily_tool(tool_type: str, framework: Framework = Framework.CREW_AI):
    """Tavily tool for the framework. tool_type: page_extractor or one of SEARCH_CONFIGS."""
    search_cls, extract_cls = _tool_classes(framework)

    if tool_type == "page_extractor":
        return extract_cls()

    if tool_type not in SEARCH_CONFIGS:
        raise ValueError(f"Unknown Tavily tool type '{tool_type}'. Available: page_extractor, {', '.join(SEARCH_CONFIGS)}")

    return search_cls(**SEARCH_CONFIGS[tool_type])


In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/tools/request_handler.py


import html
from datetime import datetime
import requests

from due_diligence_mas.mas_definition.classes import CurrentContext, Location

def get_current_context() -> CurrentContext:
    """
    Retrieves the current context of the conversation or interaction.
    It returns a information object, which can be used to provide 
    additional information to the prompt like location and timestamp.

    Returns:
        location (Location): The location of the current context
        date_time (Date): The date and time of the current context
    """
    
    current_date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        resp = requests.get("https://ipinfo.io/json")
        data = resp.json()
        location = Location(
            city=data.get("city"),
            timezone=data.get("timezone"),
            country=data.get("country"),
            region=data.get("region")
        )
    
    except Exception as e:
        location = "Unknown"

    return CurrentContext(
        location=location,
        date_time=current_date_time
        )

if __name__ == "__main__":
    
    print(get_current_context())

### Agents: spec, adapters, analysts, debate, writer

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/dd_agent.py
from dataclasses import dataclass
from typing import Optional

from pydantic import BaseModel

from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.utils.backstory_builder import backstory_builder
from due_diligence_mas.mas_definition.utils.extract_agent import agent_limits, extract_agent


@dataclass(frozen=True)
class TaskDef:
    """Framework-agnostic task definition. `{topic}` in `description` is replaced at build time."""

    description: str
    expected_output: str
    output_schema: Optional[type[BaseModel]] = None
    bind_agent: bool = True
    tools: tuple[str, ...] = ()


class DDAgent:
    """
    Agent defined only by data: subclasses set the class attributes, the framework adapters build the objects.

    Attributes:
        name: agent name, key of the agents catalogue.
        tools: tool keys, resolved by each adapter.
        delegation: whether the agent can delegate work.
        tasks_def: task definitions by key; `build_tasks` picks one by key.
    """

    name: AgentName
    tools: tuple[str, ...] = ()
    delegation: bool = False
    tasks_def: dict[str, TaskDef] = {}

    def __init__(self, framework: Framework = Framework.CREW_AI, task="main"):
        self.framework = framework
        self.spec = extract_agent(self.name.value)
        self.backstory = backstory_builder(self.name.value)
        self.limits = agent_limits(self.name.value)
        self.tasks = []
        self.agent = self._adapter().build_agent(self, task)

    def crewai_kwargs(self) -> dict:
        """Extra crewai.Agent arguments for agents that need them."""
        return {}

    def _adapter(self):
        """Return the adapter of the selected framework, imported lazily so only that framework needs to be installed."""
        match self.framework:
            case Framework.CREW_AI:
                from due_diligence_mas.mas_definition.agents.adapters import crewai as adapter
            case Framework.LANG_GRAPH:
                from due_diligence_mas.mas_definition.agents.adapters import langgraph as adapter
            case _:
                raise ValueError(f"Unsupported framework: {self.framework}")
        return adapter

    def get_agent(self):
        return self.agent

    def get_tasks(self):
        return self.tasks

    def build_tasks(self, topic: str | dict = "", context: list | None = None, key: str = "main"):
        if key not in self.tasks_def:
            raise KeyError(f"Unknown task '{key}' for {self.name.value}. Available: {', '.join(self.tasks_def)}")
        if isinstance(topic, dict):
            topic = "\n\n".join(f"## {name.capitalize()}\n{value}" for name, value in topic.items())
        self.tasks = self._adapter().build_tasks(self, self.tasks_def[key], topic, context or [])

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/adapters/crewai.py
from crewai import Agent, Task

from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.tools.apify import get_linkedin_tool
from due_diligence_mas.mas_definition.tools.github import github_repos
from due_diligence_mas.mas_definition.tools.mcp_task import get_task_server_crewai
from due_diligence_mas.mas_definition.tools.request_handler import get_current_context
from due_diligence_mas.mas_definition.tools.tavily import get_tavily_tool
from due_diligence_mas.mas_definition.crewai_fr.utils import is_verbose
from due_diligence_mas.mas_definition.providers.llm_provider import LLM_PROVIDER
from crewai.tools import tool

TOOLS = {
    "current_context": lambda: tool(get_current_context),
    "github_repos": lambda: tool(github_repos),
    "linkedin": lambda: get_linkedin_tool("profile", Framework.CREW_AI),
    "tavily_search": lambda: get_tavily_tool("search"),
    "tavily_funding_search": lambda: get_tavily_tool("funding_search"),
    "tavily_profile_search": lambda: get_tavily_tool("profile_search"),
    "tavily_page_extractor": lambda: get_tavily_tool("page_extractor"),
}

TASK_TOOLS = {
    "mcp_tasks": lambda agent: agent.get_mcp_tools([get_task_server_crewai()]),
}


def build_agent(definition, task: str = "main") -> Agent:
    return Agent(
        role=definition.spec["name"],
        goal=definition.spec["goal"],
        backstory=definition.backstory,
        llm=LLM_PROVIDER().get_llm(),
        tools=[TOOLS[key]() for key in definition.tools],
        allow_delegation=definition.delegation,
        verbose=is_verbose("agent"),
        max_retry_limit=2,
        **definition.crewai_kwargs(),
        **definition.limits,
    )


def build_tasks(definition, task_def, topic: str, context: list) -> list[Task]:
    return [
        Task(
            description=task_def.description.replace("{topic}", topic),
            expected_output=task_def.expected_output,
            agent=definition.agent if task_def.bind_agent else None,
            output_pydantic=task_def.output_schema,
            tools=[tool for key in task_def.tools for tool in TASK_TOOLS[key](definition.agent)],
            context=context,
        )
    ]

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/adapters/langgraph.py


from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.tools.github import github_repos
from due_diligence_mas.mas_definition.tools.apify import get_linkedin_tool
from due_diligence_mas.mas_definition.tools.tavily import get_tavily_tool
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.agents.structured_output import ToolStrategy


from due_diligence_mas.mas_definition.utils.agent_names import agent_slug
from due_diligence_mas.mas_definition.tools.request_handler import get_current_context
from due_diligence_mas.mas_definition.providers.llm_provider import LLM_PROVIDER

TOOLS = {
    "current_context": lambda: tool(get_current_context),
    "github_repos": lambda: tool(github_repos),
    "linkedin": lambda: get_linkedin_tool(framework=Framework.LANG_GRAPH),
    "tavily_search": lambda: get_tavily_tool("search", Framework.LANG_GRAPH),
    "tavily_funding_search": lambda: get_tavily_tool("funding_search", Framework.LANG_GRAPH),
    "tavily_profile_search": lambda: get_tavily_tool("profile_search", Framework.LANG_GRAPH),
    "tavily_page_extractor": lambda: get_tavily_tool("page_extractor", Framework.LANG_GRAPH),
}


def _resolve_tools(keys):
    missing = [k for k in keys if k not in TOOLS]
    if missing:
        raise NotImplementedError(f"No LangChain implementation yet for tools: {', '.join(missing)}")
    return [TOOLS[k]() for k in keys]


def _limit_middleware(definition):
    """Cap tool calls per run with the agent's max_iter, as CrewAI does; once reached the model must answer with what it has."""
    max_iter = definition.limits.get("max_iter")
    return [ToolCallLimitMiddleware(run_limit=max_iter)] if max_iter else []


def build_agent(definition, task):
    
    output_schema = definition.tasks_def[task].output_schema if definition.tasks_def[task] else None

    return create_agent(
        model=LLM_PROVIDER().get_chat_model(),
        tools=_resolve_tools(definition.tools),
        system_prompt=definition.backstory,
        response_format=ToolStrategy(output_schema) if output_schema else None,
        middleware=_limit_middleware(definition),
        name=agent_slug(definition.spec["name"]),
    ).with_config({"recursion_limit": 4 * (definition.limits.get("max_iter") or 10) + 20})


def build_tasks(definition, task_def, topic: str, context: list) -> list[dict]:
    """A LangGraph task is the user message sent to the agent; `context` holds previous outputs as text."""
    parts = [f"{task_def.description.replace('{topic}', topic)}"]
    parts.append(f"Goal: {definition.spec['goal']}")
    parts.append(f"Expected output:\n{task_def.expected_output}")
    if context:
        parts.append("Context:\n" + "\n\n".join(str(c) for c in context))
    return [{"role": "user", "content": "\n\n".join(parts)}]


In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/analysis_reviewer.py
from due_diligence_mas.mas_definition.agents.analysts.analyst_supervisor import AnalystSupervisor
from due_diligence_mas.mas_definition.agents.dd_agent import TaskDef
from due_diligence_mas.mas_definition.classes.company import AnalysisReview
from due_diligence_mas.mas_definition.constants.review import ANALYSIS_REVIEW_CRITERIA, TASK_CREATION_INSTRUCTIONS


class AnalysisReviewer_Agent(AnalystSupervisor):
    """The supervisor acting as a regular agent: scores the aggregated analysis and, if it decides so, creates follow-up tasks with the task manager tools."""

    delegation = False

    tasks_def = {
        "review": TaskDef(
            description=f"{ANALYSIS_REVIEW_CRITERIA}\n\nYou have task manager tools. {TASK_CREATION_INSTRUCTIONS}\n\n{{topic}}",
            expected_output="The analysis review: completeness score, missing items and a short summary.",
            output_schema=AnalysisReview,
            tools=("mcp_tasks",),
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/analyst_supervisor.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class AnalystSupervisor(DDAgent):
    name = AgentName.ANALYST_SUPERVISOR
    delegation = True

    tasks_def = {
        "main": TaskDef(
            description="Considering the company profile in the context.",
            expected_output="A markdown analysis report with one section per specialist, keeping their findings unchanged, followed by 'Resolved conflicts' and 'Missing information' sections.",
            bind_agent=False,
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/commercial_analyst.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.research import CONTEXT_FIRST

FINANCIAL_SOURCES = [
    "Companies House, UK annual accounts (https://find-and-update.company-information.service.gov.uk/): open the company, Filing history > Accounts, and the PDF of the relevant period. "
    "Record turnover, profit or loss, balance sheet, the financial year-end and whether the accounts cover the company or the group; small filings may omit the income statement.",
    "DATA INPI, French annual accounts (https://data.inpi.fr/recherche_avancee/entreprises): search by nine-digit Siren, or by name and location, then 'Documents associés' > 'comptes annuels'. "
    "Documents need a free account: if not authenticated, record the company record URL and flag the accounts for manual retrieval.",
    "Tech.eu Funding Explorer, European rounds (https://funding.tech.eu/companies): open the matching profile, confirm country and website, and capture date, stage, amount, investors and cited source of each round. "
    "Keep equity, debt and grants distinct and verify the original currency in the cited announcement.",
    "SEC EDGAR, U.S. private offerings (https://www.sec.gov/edgar/search/): search the issuer's legal name or CIK, forms D and D/A, section 13 'Offering and Sales Amounts' ('Total Amount Sold', filing date). "
    "Use the latest amendment of an offering and never count original and amended filings as separate rounds.",
    "FinSMEs, funding announcements (https://www.finsmes.com/): search site:finsmes.com \"<company name>\" funding, adding city, sector, founder or lead investor if ambiguous; "
    "capture amount, currency, stage, investors and date, and cross-check with a company or investor announcement.",
    "Company, investor and customer websites, for traction: News, Press, Customers, Case Studies pages, or site:<company-domain> (\"revenue\" OR \"ARR\" OR \"customers\"), and site:<customer-domain> \"<startup name>\" for known customers. "
    "Keep documented adoption separate from disclosed paying customers, revenue, ARR or churn.",
]


def _numbered(items: list[str]) -> str:
    return " ".join(f"{i}) {s}" for i, s in enumerate(items, 1))


class CommercialAnalyst_Agent(DDAgent):
    name = AgentName.COMMERCIAL_ANALYST
    tools = ("tavily_search", "tavily_funding_search", "tavily_page_extractor",)

    tasks_def = {
        "main": TaskDef(
            description=(
                "Review the commercial aspects of the target company: {topic}. "
                f"{CONTEXT_FIRST} "
                "Cover market, competitors, pricing, revenue model, go-to-market, traction, financial figures and funding; product/technology and team belong to other specialists. "
                "For financial and funding data, first identify the registered legal name, country and registration number, and search the official registers by number. "
                "If you search by brand name, confirm the match against the company's website, location or registered office, otherwise mark the figures as to be confirmed. "
                "Use these free sources, with the search and page extractor tools (Crunchbase, Dealroom and Openapi need a licence: do not guess their data): "
                f"{_numbered(FINANCIAL_SOURCES)} "
                "For each finding record metric, value, currency or unit, period or event date, the direct document or article URL, and whether it is filed, announced or estimated."
            ),
            expected_output=(
                "A markdown report with: a short summary; "
                "a 'Legal entity' line (legal name, country, registration number, how the match was confirmed, confidence); "
                "the findings as a list, each with its source URL, access date, metric, value, currency or unit, period or event date, "
                "status (verified, company claim or to be confirmed) and, for figures, filed, announced or estimated; "
                "sections for Market and competitors, Financials and revenue, Funding rounds and investors, and Commercial traction; "
                "and a 'Missing information' list of the items to request in the data room."
            ),
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/company_profiler.py
from due_diligence_mas.mas_definition.classes import CompanyProfile
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class CompanyProfiler_Agent(DDAgent):
    name = AgentName.COMPANY_PROFILER
    tools = ("tavily_profile_search", "tavily_page_extractor", "current_context",)

    tasks_def = {
        "main": TaskDef(
            description=(
                "Profile the following target company: {topic}. "
                "Extract all possible informations through websites, using also sitemaps to get all relevant pages. "
                "Always try the company's pages that usually hold key facts (about, team, company, careers, contact, footer links to GitHub and LinkedIn) before concluding that an item is missing. "
                "If the work is delegated more than once, the final answer must cumulate the facts of ALL the executions, in particular the earliest ones: never report only the result of the last execution, and never drop an entry already collected. "
                "If a follow-up request lists items already found, do not repeat them: search only the missing ones, and report each item you still cannot find as 'missing (not found after search)'."
            ),
            expected_output=(
                "English, no markdown, in four fields: generic, tech, commercial, team. "
                "Each field is a list with one entry per fact or gap. "
                "Be exhaustive: include every fact collected, without summarizing or merging entries; there is no length limit. "
                "Every fact entry has the form: fact | source URL | access date (YYYY-MM-DD) | verified, claimed or to be confirmed. "
                "The source must be a URL, never only a publication name.\n"
                "generic: official website, what the company does, sector, headquarters, founding year, products or services, value proposition, target customers, clients/partners/case studies, geography.\n"
                "tech (for technology review): official GitHub organization URL (only if linked from the company site or docs), documentation/API/pricing/product page URLs, technology claims found, and what is still missing.\n"
                "commercial (for commercial analysis): clients, pricing, market and traction signals, funding or press mentions found, and what is still missing.\n"
                "team (for team analysis): team page URL, LinkedIn company page URL, names and roles of visible team members, and what is still missing."
            ),
            output_schema=CompanyProfile,
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/team_analyst.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.research import ANALYST_REPORT_FORMAT, CONTEXT_FIRST


class TeamAnalyst_Agent(DDAgent):
    name = AgentName.TEAM_ANALYST
    tools = ("tavily_search", "tavily_page_extractor",)

    tasks_def = {
        "main": TaskDef(
            description=(
                "Review the founding team and organization of the target company: {topic}. "
                f"{CONTEXT_FIRST} "
                "Use the team members, team page and LinkedIn URLs found during company profiling as starting points. "
                "Use only public, verifiable information."
            ),
            expected_output=ANALYST_REPORT_FORMAT,
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/analysts/tech_reviewer.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.research import ANALYST_REPORT_FORMAT, CONTEXT_FIRST


class TechReviewer_Agent(DDAgent):
    name = AgentName.TECH_REVIEWER
    tools = ("tavily_search", "github_repos", "tavily_page_extractor",)

    tasks_def = {
        "main": TaskDef(
            description=(
                "Review the technology aspects of the target company: {topic}. "
                f"{CONTEXT_FIRST} "
                "If the company has an official GitHub organization (linked from its website or docs), use the GitHub repositories tool with that owner to gather open-source evidence; "
                "do not guess organizations, and if none is confirmed report the GitHub signals as unavailable. "
                "Check that the repositories really belong to the company and mark all GitHub data as to be confirmed."
            ),
            expected_output=ANALYST_REPORT_FORMAT,
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/debate/debate_formatter.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class DebateFormatter_Agent(DDAgent):
    name = AgentName.DEBATE_FORMATTER

    tasks_def = {
        "main": TaskDef(
            description=(
                "Prepare the debate evidence pack from the following complete analysis: {topic}. "
                "Condense it into a single structured document: a company snapshot, deduplicated evidence items each with a unique ID, source and confidence level, a clear separation of facts, assumptions and missing data, and the initial tensions within the analysis. "
                "Do not add new facts or new analysis."
            ),
            expected_output="A structured debate pack with sections: company snapshot, evidence items (ID, statement, source, confidence), facts vs assumptions vs missing data, and initial tensions.",
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/debate/debate_judge.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class DebateJudge_Agent(DDAgent):
    name = AgentName.DEBATE_JUDGE

    tasks_def = {
        "main": TaskDef(
            description=(
                "Judge the PRO/CONTRO debate on the target company, using the PRO and CONTRO cases received from the previous tasks. "
                "Weigh the strength of the PRO arguments against the severity of the CONTRO arguments, assess evidence quality, flag unsupported claims and critical missing data, decide whether risks are knowable, mitigable or deal-breaking, identify the central investment tension and whether further diligence is justified. "
                "Do not write the final memo."
            ),
            expected_output="A balanced preliminary judgement (SWOT analysis): evidence-quality assessment of both sides, central tension, risk classification, and a recommendation on further diligence, for the final memo to use.",
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/debate/opportunity_analyst.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class OpportunityAnalyst_Agent(DDAgent):
    name = AgentName.OPPORTUNITY_ANALYST

    tasks_def = {
        "main": TaskDef(
            description=(
                "Build the PRO case for the target company. "
                "Using only the formatted debate document received from the previous task, identify the strongest evidence-backed reasons to continue diligence or invest: positive signals, market timing, commercial upside, product strengths, technical defensibility, founder-market fit and traction. "
                "Reference evidence IDs for every argument, flag arguments resting on weak evidence, do not invent facts or search externally, and do not ignore missing data."
            ),
            expected_output="A structured PRO case: ranked strengths and reasons to continue diligence, each linked to evidence IDs with an evidence-strength note.",
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/debate/risk_analyst.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class RiskAnalyst_Agent(DDAgent):
    name = AgentName.RISK_ANALYST

    tasks_def = {
        "main": TaskDef(
            description=(
                "Build the CONTRO case for the target company. "
                "Using the formatted debate document and the PRO arguments received from the previous tasks, challenge weak or unsupported PRO claims and surface commercial, market, product/technical, team and financial risks, critical missing evidence and possible deal breakers. "
                "Reference evidence IDs or explicitly cite missing data for every argument; be rigorous, not merely pessimistic."
            ),
            expected_output="A structured CONTRO case: rebuttals to PRO arguments, ranked risks and deal breakers, and critical missing data, each linked to evidence IDs.",
        ),
    }

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/agents/writer/memo_writer.py
from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent, TaskDef
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName


class MemoWriter_Agent(DDAgent):
    name = AgentName.MEMO_WRITER
    tools = ("current_context",)

    tasks_def = {
        "aggregation": TaskDef(
            description="""
                    Aggregate the reports produced by the specialists (technology, commercial, team) in the context
                    into one consolidated analysis, in English.
                    Keep every finding unchanged, with its source, access date and confidence level: do not drop,
                    summarize or generalize anything. Do not add information that is not in the context.
                    Where specialists report conflicting information, keep both versions and state the conflict.
                    {topic}
                """,
            expected_output="""
                    A Markdown analysis report with one section per specialist (Product and Technology,
                    Commercial Analysis, Team and Organization), each containing all of that specialist's findings,
                    followed by:
                    ## Resolved and Unresolved Conflicts
                    ## Missing Information
                    ## Sources
                    Numbered list of all the sources cited by the specialists.
                """,
        ),
        "memo": TaskDef(
            description="""
                    Write the due diligence memo from the analysis and the debate below.
                    First call the current context tool: it returns the user's country and the current date and time.
                    - Language: write the whole memo, section titles included, in the main language of that country
                      (the tool gives the country code). If the country is unknown, write in English.
                      Translate the text only: names, figures, URLs and source titles stay unchanged.
                    - Dates: the current date from the tool is "today". Use it for the memo date and for every
                      relative reference ("as of today", "X months ago", "recent", "upcoming"), by comparing the
                      dates of the sources with it. Never assume a different date.
                    The analysis is the primary source: carry all of its findings into the memo, with their figures,
                    names, dates, sources and confidence level, and do not shorten them to save space (no length limit).
                    The debate is only summarized. Add nothing that is not in the input.
                    {topic}
                """,
            expected_output="""
                    A due diligence memo in Markdown, written as a flowing narrative that a person reads from top to bottom:
                    connected paragraphs that explain the findings and how they relate, not a list of bullet points.
                    Use tables only where they make data clearer (offering, customers, competitors, team members, risks).
                    The language and the date follow the current context, as described in the task.

                    Open with the title "Due Diligence Memo — [Company]", today's date and a line on the scope
                    (areas reviewed, evidence cutoff, exclusions). Then cover, in the order that reads best:
                    - an executive summary (about 250 words): the business, the material findings, the preliminary conclusion;
                    - the company profile, the product and technology, the commercial picture and the team, each in
                      depth and with the sources cited inline as [1], [2]; add a section for any other topic the
                      analysis covers;
                    - a short summary of the debate (PRO case, CONTRO case, the judge's conclusion), or a note that
                      the debate was not performed;
                    - the material risks, with evidence, potential impact and the follow-up needed;
                    - the information gaps and limits of the evidence, with the priority questions;
                    - the preliminary conclusion and next steps: whether the evidence supports going on, going on
                      under conditions, or pausing, and why;
                    - the numbered list of sources, with title, URL and publication or access date.

                    Along the way:
                    - say whether each statement is verified, a company claim, an estimate or still to be confirmed,
                      in the flow of the text and not as a label on every line;
                    - never invent facts or references; say when data is not available;
                    - missing evidence does not prove that a risk does not exist;
                    - keep allegations apart from established findings.
                """,
        ),
    }

    def build_tasks(self, topic: str = "", context: list = [], key: str = "memo"):
        super().build_tasks(topic, context, key)

## 3. MCP

The task manager MCP server. The analysis reviewer creates follow-up tasks through it. It has two tools: `get_users` and `create_task`. Tasks are saved as JSON in `mcp/tasks/`.

`mcp_task.py` starts the server as a stdio subprocess of the current Python. Each agent run starts its own subprocess.

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mcp/models.py
from enum import Enum
from uuid import UUID, uuid4

from pydantic import BaseModel, Field


class Priority(Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class Task(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    assignee: UUID
    description: str
    priority: Priority

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mcp/repository.py
from pathlib import Path

from due_diligence_mas.mcp.models import Task

TASKS_DIR = Path(__file__).resolve().parent / "tasks"


def save_task(task: Task) -> Path:
    """Store the task as <task_id>.json in the tasks folder."""
    TASKS_DIR.mkdir(parents=True, exist_ok=True)
    path = TASKS_DIR / f"{task.id}.json"
    path.write_text(task.model_dump_json(indent=2))
    return path

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mcp/users.py
from enum import Enum
from uuid import UUID, uuid4

from pydantic import BaseModel, Field


class UserRole(Enum):
    TECH = "tech"
    COMMERCIAL = "commercial"


class User(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    name: str = Field(description="Name of the user")
    role: UserRole = Field(description="Role of the user that identifies his knowledge")


def get_mock_users() -> list[User]:
    """Mock users with fixed ids: tasks reference them, so they must be stable across calls."""
    return [
        User(id=UUID("cd9a5f5e-e0c3-45d0-98a0-026b1e3cf619"), name="Daniel", role=UserRole.TECH),
        User(id=UUID("7b0e3c1a-5d42-4f6e-9a18-3c2f8d6b4e57"), name="Luis", role=UserRole.COMMERCIAL),
    ]

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mcp/task_manager.py
import asyncio
import os
from uuid import UUID

from mcp.server.fastmcp import FastMCP

from due_diligence_mas.mcp.models import Priority, Task
from due_diligence_mas.mcp.repository import TASKS_DIR, save_task
from due_diligence_mas.mcp.users import User, get_mock_users
from due_diligence_mas.utils.logger import log_mcp_task, setup_logger

TRANSPORT = os.getenv("MCP_TRANSPORT", "stdio")
HOST = os.getenv("MCP_HOST", "127.0.0.1")
PORT = int(os.getenv("MCP_PORT", "8200"))

mcp = FastMCP("Task manager", host=HOST, port=PORT)


@mcp.tool()
def get_users() -> list[User]:
    """
        Get available users of task manager system

        Returns:
            List of users
    """
    users = get_mock_users()
    log_mcp_task("get_users", "done", count=len(users))
    return users


@mcp.tool()
def create_task(assignee: str, description: str, priority: str) -> Task:
    """
        Create a task and save it as a JSON file named after the task id

        Args:
            assignee: UUID of the user the task is assigned to
            description: Task description
            priority: HIGH, MEDIUM or LOW

        Returns:
            The created task
    """
    log_mcp_task("create_task", "start", assignee=assignee, priority=priority)
    task = Task(
        assignee=UUID(assignee),
        description=description,
        priority=Priority(priority.upper()),
    )
    path = save_task(task)
    log_mcp_task("create_task", "saved", task_id=task.id, path=path)
    return task


def main():
    """Start the MCP server on the configured transport. Console logs go to stderr, stdout stays free for stdio."""
    setup_logger("mcp_task", enable_console=True)
    tools = [t.name for t in asyncio.run(mcp.list_tools())]
    if TRANSPORT == "stdio":
        log_mcp_task("server", "starting", transport=TRANSPORT, port="n/a (stdio)", tools=tools, tasks_dir=TASKS_DIR)
    else:
        log_mcp_task("server", "starting", transport=TRANSPORT, url=f"http://{HOST}:{PORT}{mcp.settings.streamable_http_path}", tools=tools, tasks_dir=TASKS_DIR)
    mcp.run(transport=TRANSPORT)

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/tools/mcp_task.py
import sys

TASK_SERVER_COMMAND = sys.executable
TASK_SERVER_ARGS = ["-c", "from due_diligence_mas.mcp.task_manager import main; main()"]


async def get_task_tools():
    """LangChain tools (get_users, create_task) exposed by the task manager MCP server. MCP tools are async only."""
    from langchain_mcp_adapters.client import MultiServerMCPClient

    client = MultiServerMCPClient({"task_manager": {"transport": "stdio", "command": TASK_SERVER_COMMAND, "args": TASK_SERVER_ARGS}})
    return await client.get_tools()


def get_task_server_crewai():
    """CrewAI stdio config of the task manager MCP server."""
    from crewai.mcp import MCPServerStdio

    return MCPServerStdio(command=TASK_SERVER_COMMAND, args=TASK_SERVER_ARGS)

Colab fix. `stdio_client` sends the server stderr to `sys.stderr`, and in a notebook that stream has no file descriptor (`UnsupportedOperation: fileno`). This cell points it to a log file. It covers the LangChain adapter and CrewAI, which use the same function.

In [ ]:
import mcp.client.stdio as mcp_stdio

MCP_STDERR_LOG = ROOT / "log" / "mcp_server_stderr.log"
MCP_STDERR_LOG.parent.mkdir(parents=True, exist_ok=True)
mcp_stdio.stdio_client.__wrapped__.__defaults__ = (open(MCP_STDERR_LOG, "a"),)

Check that the server starts over stdio and lists its tools. Top-level `await` works in Colab.

In [ ]:
from due_diligence_mas.mas_definition.tools.mcp_task import get_task_tools

tools = await get_task_tools()
print([t.name for t in tools])

## 4. CrewAI

Crews (profiler, analysts, review, debate, writer) and the flow that chains them.

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/utils/__init__.py
from due_diligence_mas.mas_definition.crewai_fr.utils.verbose import is_verbose

__all__ = ["is_verbose"]

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/utils/verbose.py
import os
from typing import Literal

from dotenv import load_dotenv

load_dotenv()

VerboseScope = Literal["agent", "crew", "flow"]


def is_verbose(scope: VerboseScope) -> bool:
    """Verbosity of agents, crews or flows, from AGENT_VERBOSE, CREW_VERBOSE or FLOW_VERBOSE (default: true)."""
    return os.getenv(f"{scope.upper()}_VERBOSE", "true").strip().lower() in ("1", "true", "yes", "on")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/base.py
from typing import Optional
import time

from crewai import Agent, Crew, Process, Task
from crewai.llms.base_llm import BaseLLM

from due_diligence_mas.mas_definition.agents.dd_agent import DDAgent
from due_diligence_mas.mas_definition.crewai_fr.utils import is_verbose
from due_diligence_mas.mas_definition.utils.llm_cost import summarize_cost
from due_diligence_mas.utils.logger import log_crew


class BaseCrew:
    """
    Common skeleton of the crews: subclasses implement `_setup` and register their agents with `add`.
    Agents are added in execution order.
    """

    name: str = "Crew"
    process: Process = Process.sequential
    manager: Optional[DDAgent] = None

    def __init__(self):
        self.crew_agents: list[Agent] = []
        self.crew_tasks: list[Task] = []
        self._setup()

    def _setup(self):
        raise NotImplementedError

    def add(self, agent: DDAgent, topic, *, context: Optional[list[Task]] = None, key: str = "main") -> DDAgent:
        """Register the agent and build its tasks; the task context is decided by the calling crew."""
        agent.build_tasks(topic, context=context, key=key)
        self.crew_agents.append(agent.get_agent())
        self.crew_tasks.extend(agent.get_tasks())
        return agent

    def _crew_kwargs(self) -> dict:
        kwargs = {}
        if self.manager is not None:
            kwargs["manager_agent"] = self.manager.get_agent()
        return kwargs

    def build_crew(self):
        self.crew = Crew(
            name=self.name,
            agents=self.crew_agents,
            tasks=self.crew_tasks,
            process=self.process,
            tracing=True,
            verbose=is_verbose("crew"),
            **self._crew_kwargs(),
        )

    def run_crew(self):
        if not hasattr(self, "crew"):
            self.build_crew()
        log_crew(self.name, "start", agents=len(self.crew_agents), tasks=len(self.crew_tasks))
        start = time.perf_counter()
        try:
            result = self.crew.kickoff()
        except Exception as e:
            log_crew(self.name, "failed", elapsed_s=round(time.perf_counter() - start, 2), error=e)
            raise
        self.metrics = self._collect_metrics(time.perf_counter() - start)
        log_crew(
            self.name, "end", elapsed_s=self.metrics["duration_s"], llm_calls=self.metrics["llm_calls"],
            tokens=self.metrics["tokens"]["total"], cost_usd=self.metrics["cost_usd"]["total"],
        )
        return result

    def _usage_by_model(self) -> dict:
        """Tokens by model, from the LLM of every agent: the same source of `crew.token_usage`, which is not split by model."""
        agents = [*self.crew.agents, *([self.crew.manager_agent] if self.crew.manager_agent else [])]
        models: dict[str, dict[str, int]] = {}
        for llm in {id(agent.llm): agent.llm for agent in agents}.values():
            if isinstance(llm, BaseLLM):
                usage = llm.get_token_usage_summary()
                totals = models.setdefault(llm.model, {"prompt_tokens": 0, "completion_tokens": 0})
                totals["prompt_tokens"] += usage.prompt_tokens
                totals["completion_tokens"] += usage.completion_tokens
        return models

    def _collect_metrics(self, duration_s: float) -> dict:
        usage = self.crew.token_usage
        models = self._usage_by_model()
        return {
            "name": self.name,
            "duration_s": round(duration_s, 2),
            "llm_calls": usage.successful_requests,
            "tokens": {
                "prompt": usage.prompt_tokens,
                "completion": usage.completion_tokens,
                "total": usage.total_tokens,
            },
            "models": models,
            "cost_usd": summarize_cost(models),
        }

    def print_crew_info(self):
        print("Crew Agents:")
        for agent in self.crew_agents:
            print(f"- {agent.role}")

        print("\nCrew Tasks:")
        for task in self.crew_tasks:
            print(f"- {task.description}")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/profiler.py
from crewai import Process

from due_diligence_mas.mas_definition.agents.analysts.analyst_supervisor import AnalystSupervisor
from due_diligence_mas.mas_definition.agents.analysts.company_profiler import CompanyProfiler_Agent
from due_diligence_mas.mas_definition.classes import CompanyInfo
from due_diligence_mas.mas_definition.crewai_fr.crews.base import BaseCrew


class ProfilerCrew(BaseCrew):
    """
    Crew that profiles the target company (hierarchical, supervisor as manager).
    """

    name = "Profiler Crew"
    process = Process.hierarchical

    def __init__(self, company_info: CompanyInfo):
        self.topic = company_info.describe()
        super().__init__()

    def _setup(self):
        self.manager = AnalystSupervisor()
        self.profiler = self.add(CompanyProfiler_Agent(), self.topic)

    def build_crew(self):
        """Build the crew, then strip tools from the tasks: the manager runs every task and would otherwise use the profiler's tools itself. __dict__ bypasses the validator, which would copy the tools back."""
        super().build_crew()

        for task in self.crew.tasks:
            task.__dict__["tools"] = []

    def run_crew(self):
        result = super().run_crew()
        self._log_executions(result)
        return result

    @staticmethod
    def _entry_counts(output) -> str:
        profile = getattr(output, "pydantic", None)
        if not profile:
            return f"no structured profile (raw chars: {len(output.raw)})"
        counts = {f: len(getattr(profile, f) or []) for f in ("generic", "tech", "commercial", "team")}
        return f"{counts} total={sum(counts.values())}"

    def _log_executions(self, result):
        """Print the entry counts of the manager's final profile."""
        print(f"\n{'-'*80}")
        print(f"  Final profile (manager): {self._entry_counts(result)}")
        print(f"{'-'*80}\n")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/analyst.py
from due_diligence_mas.mas_definition.agents.analysts.commercial_analyst import CommercialAnalyst_Agent
from due_diligence_mas.mas_definition.agents.analysts.team_analyst import TeamAnalyst_Agent
from due_diligence_mas.mas_definition.agents.analysts.tech_reviewer import TechReviewer_Agent
from due_diligence_mas.mas_definition.agents.writer.memo_writer import MemoWriter_Agent
from due_diligence_mas.mas_definition.classes import ProfiledCompany
from due_diligence_mas.mas_definition.crewai_fr.crews.base import BaseCrew


class AnalystCrew(BaseCrew):
    """
    Sequential crew: the specialists run first, then the Memo Writer aggregates their reports.
    """

    name = "Analyst Crew"

    def __init__(self, company: ProfiledCompany, required_agents: list[str] | None = None):
        self.company = company
        self.required_agents = required_agents or []
        super().__init__()

    def _is_required(self, agent_cls) -> bool:
        return not self.required_agents or agent_cls.name.value in self.required_agents

    def _setup(self):
        """Add the specialists, then the memo writer that aggregates all their outputs."""
        specialists = [
            (TechReviewer_Agent, "tech"),
            (CommercialAnalyst_Agent, "commercial"),
            (TeamAnalyst_Agent, "team"),
        ]
        added = [
            self.add(agent_cls(), self.company.topic_for(area))
            for agent_cls, area in specialists
            if self._is_required(agent_cls)
        ]

        specialist_tasks = [task for agent in added for task in agent.get_tasks()]
        self.memo_writer = self.add(
            MemoWriter_Agent(), self.company.profile_topic(), context=specialist_tasks, key="aggregation"
        )

    def run_crew(self) -> str:
        """Returns the consolidated analysis in markdown."""
        return super().run_crew().raw

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/review.py
from due_diligence_mas.mas_definition.agents.analysts.analysis_reviewer import AnalysisReviewer_Agent
from due_diligence_mas.mas_definition.classes import ProfiledCompany
from due_diligence_mas.mas_definition.classes.company import AnalysisReview
from due_diligence_mas.mas_definition.crewai_fr.crews.base import BaseCrew


class AnalysisReviewCrew(BaseCrew):
    """
    Crew that scores the completeness of the aggregated analysis; the reviewer may create follow-up tasks through the task manager MCP tools.
    """

    name = "Analysis Review Crew"

    def __init__(self, company: ProfiledCompany, analysis: str):
        generic = "\n".join(company.company_profile.generic or []) if company.company_profile else ""
        self.topic = f"COMPANY: {company.company_info.describe()}\n\nPROFILE:\n{generic}\n\nAGGREGATED ANALYSIS:\n{analysis}"
        super().__init__()

    def _setup(self):
        self.reviewer = self.add(AnalysisReviewer_Agent(), self.topic, key="review")

    def run_crew(self) -> AnalysisReview:
        """Returns the structured review."""
        return super().run_crew().pydantic

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/debate.py
from due_diligence_mas.mas_definition.agents.debate.debate_formatter import DebateFormatter_Agent
from due_diligence_mas.mas_definition.agents.debate.debate_judge import DebateJudge_Agent
from due_diligence_mas.mas_definition.agents.debate.opportunity_analyst import OpportunityAnalyst_Agent
from due_diligence_mas.mas_definition.agents.debate.risk_analyst import RiskAnalyst_Agent
from due_diligence_mas.mas_definition.crewai_fr.crews.base import BaseCrew


class DebateCrew(BaseCrew):
    """
    Opportunity vs. risk debate, optionally preceded by a formatter, closed by the judge.
    """

    name = "Debate Crew"

    def __init__(self, args: dict | None = None, need_formatter: bool = True):
        self.topic = args or {}
        self.need_formatter = need_formatter
        super().__init__()

    def _setup(self):
        formatter_tasks = []
        if self.need_formatter:
            formatter = self.add(DebateFormatter_Agent(), self.topic)
            formatter_tasks = formatter.get_tasks()

        opportunity = self.add(OpportunityAnalyst_Agent(), self.topic, context=formatter_tasks)
        risk = self.add(
            RiskAnalyst_Agent(), self.topic, context=formatter_tasks + opportunity.get_tasks()
        )
        self.add(
            DebateJudge_Agent(), self.topic, context=opportunity.get_tasks() + risk.get_tasks()
        )

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/crews/writer.py
from due_diligence_mas.mas_definition.agents.writer.memo_writer import MemoWriter_Agent
from due_diligence_mas.mas_definition.crewai_fr.crews.base import BaseCrew


class MemoWriterCrew(BaseCrew):
    """
    Crew that writes the final memo.
    """

    name = "Memo writer Crew"

    def __init__(self, args: dict | None = None):
        self.topic = args or {}
        super().__init__()

    def _setup(self):
        self.memo_writer = self.add(MemoWriter_Agent(), self.topic, key="memo")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/crewai_fr/flows/due_diligence.py
import os

from crewai.flow import Flow, start, listen, router, or_, HumanFeedbackPending, HumanFeedbackProvider, human_feedback
from due_diligence_mas.mas_definition.classes import CompanyInfo, ProfiledCompany, CompanyProfile
from due_diligence_mas.mas_definition.classes.company import AnalysisReview
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.crewai_fr.crews.analyst import AnalystCrew
from due_diligence_mas.mas_definition.crewai_fr.crews.debate import DebateCrew
from due_diligence_mas.mas_definition.crewai_fr.crews.profiler import ProfilerCrew
from due_diligence_mas.mas_definition.crewai_fr.crews.review import AnalysisReviewCrew
from due_diligence_mas.mas_definition.crewai_fr.crews.writer import MemoWriterCrew
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.utils.execution_db import ExecutionStatus, create_execution, update_execution
from due_diligence_mas.mas_definition.utils.llm_cost import summarize_cost
from due_diligence_mas.mas_definition.crewai_fr.utils import is_verbose
from due_diligence_mas.mas_definition.utils.request_handling import request_company_info
from due_diligence_mas.utils.logger import log_crew
from pydantic import BaseModel, Field
from crewai.flow.persistence import SQLiteFlowPersistence


from functools import wraps
from typing import Optional

FLOW_DB_PATH = os.getenv("CREWAI_FLOW_DB_PATH", "due_diligence_crewai.db")
APPROVAL_ANSWERS = {"y", "yes", "proceed", "approve", "approved"}

def track_crew(run_crew):
    """Store the crew in progress on the execution row, then the metrics when it ends."""
    @wraps(run_crew)
    def wrapper(self, crew):
        update_execution(self.state.id, current_crew=crew.name)
        result = run_crew(self, crew)
        update_execution(self.state.id, metrics=self._execution_metrics())
        return result
    return wrapper

class DueDiligenceFlow_State(BaseModel):
    
    company_info: Optional[CompanyInfo] = None
    
    profile: Optional[CompanyProfile] = None
    
    analysis_results: Optional[str] = None
    analysis_review: Optional[AnalysisReview] = None
    crews_metrics: list[dict] = Field(default_factory=list)
    
    debate_results: Optional[str] = None  
    final_report: Optional[str] = None
    
class RestFeedbackProvider(HumanFeedbackProvider):
    def request_feedback(self, context, flow) -> str:
        raise HumanFeedbackPending(context=context)

class DueDiligenceFlow(Flow[DueDiligenceFlow_State]):
    
    @start()
    def initialize_flow(self):
        """
        Initialize the due diligence flow.
        """
        if self.state.company_info is None:
            self.state.company_info = request_company_info()
            
        log_crew("flow", "initialized", company=self.state.company_info.company_name)
        
        return self.state.company_info
    
    @listen(initialize_flow)
    def profile_company(self):
        """ 
            Profile the target company.
        """
        
        log_crew("flow", "profile_company")
                
        analyst_crew = ProfilerCrew(company_info=self.state.company_info)
                    
        report = self._run_crew(analyst_crew)
        profile = report.pydantic
        log_crew("flow", "profile_completed", profiled=profile is not None)
        self.state.profile = profile
        
        return profile
        
    @listen(profile_company)
    def analyze_company(self):
        """
        Analyze the target company.
        """
        log_crew("flow", "analyze_company")
        
        analyst_crew = AnalystCrew(
            company=ProfiledCompany(
                company_info=self.state.company_info,
                company_profile=self.state.profile
            ),
            required_agents=[
                    AgentName.TECH_REVIEWER.value, 
                    AgentName.COMMERCIAL_ANALYST.value,
                    AgentName.TEAM_ANALYST.value
                ]
            )
            
        report = self._run_crew(analyst_crew)
        self.state.analysis_results = report

        return self.state.analysis_results
    
    @listen(analyze_company)
    def review_analysis(self):
        """
        Completeness gate: score of the analysis, with follow-up tasks created by the reviewer if needed.
        """
        log_crew("flow", "review_analysis")

        review_crew = AnalysisReviewCrew(
            company=ProfiledCompany(company_info=self.state.company_info, company_profile=self.state.profile),
            analysis=self.state.analysis_results,
        )
        self.state.analysis_review = self._run_crew(review_crew)
        log_crew("flow", "analysis_reviewed", score=self.state.analysis_review.score, missing=len(self.state.analysis_review.missing))

        return self.state.analysis_results

    @listen(review_analysis)
    @human_feedback(     
          message="Please review the analysis and proceed or end the execution",
          provider=RestFeedbackProvider() 
    )
    def ask_human_review(self, analysis_results):
        return analysis_results
    
    @router(ask_human_review)
    def human_decision_router(self):
        feedback = self.last_human_feedback.feedback if self.last_human_feedback else ""
        is_approved = feedback.strip().lower() in APPROVAL_ANSWERS

        log_crew("flow", "human_approval", decision="debate" if is_approved else "final_report")
        return "debate" if is_approved else "final_report"
    
    @listen("debate")
    def run_debate(self):
        log_crew("flow", "run_debate")
        
        debate_crew = DebateCrew(
                args={
                    "analysis": self.state.analysis_results
                }
            )
            
        result = self._run_crew(debate_crew)
        self.state.debate_results = result.raw
        
        return result
    
    @listen(or_("final_report", run_debate))
    def generate_final_report(self):
        log_crew("flow", "generate_final_report")
        
        if not self.state.analysis_results:
            raise ValueError("Analysis not produced")
        
        memo_crew = MemoWriterCrew(
            args={
                    "analysis": self.state.analysis_results,
                    "debate": self.state.debate_results if self.state.debate_results else "Debate not performed"
            }
        )
          
        result = self._run_crew(memo_crew)
        self.state.final_report = result.raw
        update_execution(
            self.state.id,
            status=ExecutionStatus.COMPLETED,
            current_node=None,
            current_crew=None,
            metrics=self._execution_metrics(),
            memo_markdown=self.state.final_report,
        )
        log_crew("flow", "completed")
    
        return self.state.final_report

    @track_crew
    def _run_crew(self, crew):
        result = crew.run_crew()
        self.state.crews_metrics.append(crew.metrics)
        return result

    def _execution_metrics(self) -> dict:
        """Totals over the crews (human approval time is outside the crews, so it is not counted) and the split by crew."""
        crews = self.state.crews_metrics
        prompt = sum(c["tokens"]["prompt"] for c in crews)
        completion = sum(c["tokens"]["completion"] for c in crews)
        models: dict[str, dict[str, int]] = {}
        for crew in crews:
            for model, usage in crew["models"].items():
                totals = models.setdefault(model, {"prompt_tokens": 0, "completion_tokens": 0})
                totals["prompt_tokens"] += usage["prompt_tokens"]
                totals["completion_tokens"] += usage["completion_tokens"]
        return {
            "total_time_s": round(sum(c["duration_s"] for c in crews), 2),
            "llm_calls": sum(c["llm_calls"] for c in crews),
            "tokens": {"prompt": prompt, "completion": completion, "total": prompt + completion},
            "cost_usd": summarize_cost(models),
            "crews": crews,
            "tracing": {"crewai_tracing_enabled": os.getenv("CREWAI_TRACING_ENABLED", "").lower() == "true"},
        }
    
def _close_execution(flow: DueDiligenceFlow, thread_id: str, result):
    """A paused flow returns HumanFeedbackPending: mark the execution as awaiting review."""
    if isinstance(result, HumanFeedbackPending):
        update_execution(
            thread_id,
            status=ExecutionStatus.AWAITING_REVIEW,
            current_node="ask_human_review",
            current_crew=None,
            metrics=flow._execution_metrics(),
            pending_review={
                "analysis": flow.state.analysis_results,
                "analysis_review": flow.state.analysis_review.model_dump() if flow.state.analysis_review else None,
                "options": ["proceed", "end"],
            },
        )
    return result

def run_due_diligence_flow(company_info: CompanyInfo, thread_id: str):
    """ 
        One instance per request is needed cause CrewAI is statefull
    """
    create_execution(thread_id, Framework.CREW_AI.value, company_info.company_name)
    flow = DueDiligenceFlow(
        tracing=os.getenv("CREWAI_TRACING_ENABLED", "").lower() == "true",
        suppress_flow_events=not is_verbose("flow"),
        persistence=SQLiteFlowPersistence(FLOW_DB_PATH),
    )
    try:
        result = flow.kickoff(inputs={"id": thread_id, "company_info": company_info})
    except Exception:
        update_execution(thread_id, status=ExecutionStatus.FAILED, metrics=flow._execution_metrics())
        raise
    return _close_execution(flow, thread_id, result)
    
def resume_due_diligence_flow(decision: str, thread_id: str):
    update_execution(thread_id, status=ExecutionStatus.RUNNING, pending_review=None)
    flow = DueDiligenceFlow.from_pending(thread_id, persistence=SQLiteFlowPersistence(FLOW_DB_PATH))
    try:
        result = flow.resume(decision)
    except Exception:
        update_execution(thread_id, status=ExecutionStatus.FAILED, metrics=flow._execution_metrics())
        raise
    return _close_execution(flow, thread_id, result)

if __name__ == "__main__":
    run_due_diligence_flow(request_company_info(), "local")

## 5. LangGraph

Graph state, nodes and the compiled graph. Checkpoints are saved in the executions DB, so a run paused at the human review can resume.

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/constants/langgraph.py
import os

from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.utils.agent_names import get_agent_node_name, get_node_name

REQUEST_HANDLER_NODE = get_node_name("request_handler")
PROFILER_NODE = get_agent_node_name(AgentName.COMPANY_PROFILER)
SUPERVISOR_NODE = get_node_name("supervisor")
FAN_OUT_NODE = get_node_name("analyst_fan_out")
TECH_REVIEWER_NODE = get_agent_node_name(AgentName.TECH_REVIEWER)
COMMERCIAL_ANALYST_NODE = get_agent_node_name(AgentName.COMMERCIAL_ANALYST)
TEAM_ANALYST_NODE = get_agent_node_name(AgentName.TEAM_ANALYST)
ANALYSIS_AGGREGATOR_NODE = get_node_name("analysis_aggregator")
HUMAN_REVIEW_NODE = get_node_name("human_review")
DEBATE_FORMATTER_NODE = get_agent_node_name(AgentName.DEBATE_FORMATTER)
OPPORTUNITY_ANALYST_NODE = get_agent_node_name(AgentName.OPPORTUNITY_ANALYST)
RISK_ANALYST_NODE = get_agent_node_name(AgentName.RISK_ANALYST)
DEBATE_JUDGE_NODE = get_agent_node_name(AgentName.DEBATE_JUDGE)
MEMO_WRITER_NODE = get_agent_node_name(AgentName.MEMO_WRITER)

ANALYST_AGENTS = (AgentName.TECH_REVIEWER, AgentName.COMMERCIAL_ANALYST, AgentName.TEAM_ANALYST)
ENABLED_ANALYST_AGENTS = (AgentName.TECH_REVIEWER, AgentName.COMMERCIAL_ANALYST, AgentName.TEAM_ANALYST)

MAX_PROFILE_REVISIONS = 2


def skip_debate_formatter() -> bool:
    """True (default) when the debate formatter is skipped: the debate works directly on the analysis results."""
    return os.getenv("SKIP_DEBATE_FORMATTER", "true").strip().lower() in ("1", "true", "yes", "on")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/classes/shared_state.py
from due_diligence_mas.mas_definition.classes.company import AnalysisReview, CompanyProfile, ProfiledCompany, ProfileReview
from typing import Annotated, Optional
import operator
from pydantic import BaseModel, Field


def merge_profile(old: Optional[CompanyProfile], new: Optional[CompanyProfile]) -> Optional[CompanyProfile]:
    """Reducer: overwrite only the profile fields explicitly set by the node, keep the others."""
    if old is None or new is None:
        return new if old is None else old
    return old.model_copy(update={k: getattr(new, k) for k in new.model_fields_set})


class SupervisionState(BaseModel):
    """Control data of the supervisor: profile gate and analysis completeness gate."""

    profile_review: Optional[ProfileReview] = None
    profile_revisions: int = 0
    profile_validated: bool = False
    requested_issues: list[str] = Field(default_factory=list)
    analysis_review: Optional[AnalysisReview] = None


def merge_supervision(old: SupervisionState, new: SupervisionState) -> SupervisionState:
    """Reducer: overwrite only the fields explicitly set by the node; requested_issues accumulates."""
    update = {k: getattr(new, k) for k in new.model_fields_set}
    if "requested_issues" in update:
        update["requested_issues"] = old.requested_issues + update["requested_issues"]
    return old.model_copy(update=update)


class DueDiligence_State(ProfiledCompany):

    company_profile: Annotated[Optional[CompanyProfile], merge_profile] = None
    profile_iterations: Annotated[list[CompanyProfile], operator.add] = Field(default_factory=list)

    supervision: Annotated[SupervisionState, merge_supervision] = Field(default_factory=SupervisionState)

    analysis_results: Optional[str] = None
    human_decision: Optional[str] = None
    debate_pack: Optional[str] = None
    pro_case: Optional[str] = None
    contro_case: Optional[str] = None
    debate_results: Optional[str] = None

    final_report: Optional[str] = None

    messages: Annotated[list, operator.add] = Field(default_factory=list)

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/request_handler.py
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from due_diligence_mas.mas_definition.utils.request_handling import print_welcome_message, request_company_info

def request_handler_node(state: DueDiligence_State):
    
    print_welcome_message()
    
    company_info = state.company_info
    if company_info is None:
        company_info = request_company_info()
    
    return {"company_info": company_info}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/profiler.py
from due_diligence_mas.mas_definition.agents.analysts.company_profiler import CompanyProfiler_Agent
from due_diligence_mas.mas_definition.classes.company import CompanyProfile
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.constants.langgraph import PROFILER_NODE
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State, SupervisionState
from due_diligence_mas.utils.logger import log_node as _log

def company_profiler_node(state: DueDiligence_State):
    """Run the profiler (first pass or revision with supervisor feedback). The agent's ToolStrategy(CompanyProfile) fills `structured_response`."""
    profiler = CompanyProfiler_Agent(framework=Framework.LANG_GRAPH)
    review = state.supervision.profile_review
    revision = review is not None and not review.approved
    _log(PROFILER_NODE, "start", revision=state.supervision.profile_revisions if revision else 0)

    context = []
    if revision:
        context = [f"PREVIOUS PROFILE:\n{state.company_profile.model_dump_json(indent=2)}", f"SUPERVISOR FEEDBACK (fix these issues, keep the correct facts):\n{review.feedback}\n" + "\n".join(f"- {i}" for i in review.issues)]
    profiler.build_tasks(topic=state.company_info.describe(), context=context)

    result = profiler.get_agent().invoke({"messages": profiler.get_tasks()})

    return {
        "company_profile": result["structured_response"],
        "profile_iterations": [result["structured_response"]],
        "supervision": SupervisionState(profile_revisions=state.supervision.profile_revisions + 1),
    }


In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/supervisor.py
from pydantic import BaseModel

from due_diligence_mas.mas_definition.classes.company import CompanyProfile, ProfileReview
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.langgraph import DEBATE_FORMATTER_NODE, FAN_OUT_NODE, HUMAN_REVIEW_NODE, MAX_PROFILE_REVISIONS, OPPORTUNITY_ANALYST_NODE, PROFILER_NODE, SUPERVISOR_NODE, skip_debate_formatter
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State, SupervisionState
from due_diligence_mas.mas_definition.langgraph.node.analysis_review import review_analysis
from due_diligence_mas.utils.logger import log_node as _log
from due_diligence_mas.mas_definition.providers.llm_provider import LLM_PROVIDER
from due_diligence_mas.mas_definition.utils.backstory_builder import backstory_builder


MERGE_INSTRUCTIONS = (
    "Merge the profile iterations below into one profile. Rules:\n"
    "- keep EVERY fact of EVERY iteration: a later iteration completes the earlier ones, it never replaces them;\n"
    "- if the same fact appears more than once, keep a single entry, preferring the most complete one (source URL, access date, strongest status);\n"
    "- if iterations conflict on a fact, keep the better-sourced entry and note the conflict in the entry;\n"
    "- a gap declared in an earlier iteration is dropped only if a later iteration fills it;\n"
    "- do not invent, summarize or rewrite facts; keep the entry format: fact | source URL | access date | status."
)

REVIEW_CRITERIA = (
    "Review the company profile below as the quality gate before the specialists build on it. Approve only if:\n"
    "- generic, tech, commercial and team are all filled, with one entry per fact or declared gap;\n"
    "- every fact entry has a source URL (never only a publication name), an access date and a status;\n"
    "- the entries concern the target company (no homonyms) and are not generic or unsupported claims;\n"
    "- missing information is declared explicitly as a gap, not omitted.\n"
    "If not approved, list the concrete issues and give the profiler actionable feedback.\n"
    "Issues listed under ALREADY REQUESTED were sent to the profiler in a previous iteration: never report them again, even if still open. "
    "Report only NEW issues, and give feedback only for them. Approve if no new issue is left."
)


def debate_entry_node() -> str:
    """First node of the debate: the formatter, or the PRO analyst when the formatter is skipped."""
    return OPPORTUNITY_ANALYST_NODE if skip_debate_formatter() else DEBATE_FORMATTER_NODE


def _supervisor_llm_call(instructions: str, content: str, output_schema: type[BaseModel]):
    """Single LLM call of the supervisor: given the instructions, the content to work on and the expected output schema, return the structured output."""
    llm = LLM_PROVIDER().get_chat_model().with_structured_output(output_schema)
    return llm.invoke([
        {"role": "system", "content": backstory_builder(AgentName.ANALYST_SUPERVISOR.value)},
        {"role": "user", "content": f"{instructions}\n\n{content}"},
    ])


def _merge_iterations(iterations: list[CompanyProfile]) -> CompanyProfile:
    """Union of the profiles of all iterations into a single CompanyProfile."""
    runs = "\n\n".join(f"ITERATION {i}:\n{p.model_dump_json(indent=2)}" for i, p in enumerate(iterations, 1))
    merged = _supervisor_llm_call(MERGE_INSTRUCTIONS, runs, CompanyProfile)
    _log(SUPERVISOR_NODE, "merged", iterations=len(iterations), entries={f: len(getattr(merged, f) or []) for f in CompanyProfile.model_fields})
    return merged


def _review_profile(state: DueDiligence_State):
    """Profile gate: merges the iterations, then approves the profile or sends it back to the profiler with feedback. An issue already requested is never sent again; the gate closes when approved or when the revisions are exhausted."""
    profile = _merge_iterations(state.profile_iterations) if len(state.profile_iterations) > 1 else state.company_profile
    requested = "\n".join(f"- {i}" for i in state.supervision.requested_issues) or "- none"
    review = _supervisor_llm_call(REVIEW_CRITERIA, f"COMPANY: {state.describe()}\n\nALREADY REQUESTED:\n{requested}\n\nPROFILE:\n{profile.model_dump_json(indent=2)}", ProfileReview)
    seen = {i.strip().lower() for i in state.supervision.requested_issues}
    review.issues = [i for i in review.issues if i.strip().lower() not in seen]
    if not review.issues:
        review.approved, review.feedback = True, ""
    _log(SUPERVISOR_NODE, "profile review", approved=review.approved, new_issues=len(review.issues), already_requested=len(seen), revisions=state.supervision.profile_revisions)
    validated = review.approved or state.supervision.profile_revisions > MAX_PROFILE_REVISIONS
    return {"company_profile": profile, "supervision": SupervisionState(profile_review=review, profile_validated=validated, requested_issues=review.issues)}


def supervisor_node(state: DueDiligence_State):
    """Single supervisor node, guarantor of profiling quality and analysis completeness.

    First visit(s), after the profiler: profile gate. Visit after the aggregator: completeness score."""
    return review_analysis(state) if state.supervision.profile_validated else _review_profile(state)


def supervisor_router(state: DueDiligence_State):
    """After the analysis review go on to the human review; after the profile gate go back to the profiler or to the fan out node."""
    if state.supervision.analysis_review is not None:
        return HUMAN_REVIEW_NODE
    if not state.supervision.profile_validated:
        return PROFILER_NODE
    return FAN_OUT_NODE

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/analysts.py

from due_diligence_mas.mas_definition.agents.analysts.commercial_analyst import CommercialAnalyst_Agent
from due_diligence_mas.mas_definition.agents.analysts.team_analyst import TeamAnalyst_Agent
from due_diligence_mas.mas_definition.agents.analysts.tech_reviewer import TechReviewer_Agent
from due_diligence_mas.mas_definition.classes.company import CompanyProfile
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.constants.langgraph import COMMERCIAL_ANALYST_NODE, ENABLED_ANALYST_AGENTS, FAN_OUT_NODE, TEAM_ANALYST_NODE, TECH_REVIEWER_NODE
from due_diligence_mas.mas_definition.utils.agent_names import get_agent_node_name
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from langgraph.types import Send
import time
from due_diligence_mas.utils.logger import log_node as _log


def analyst_fan_out_node(state: DueDiligence_State):
    
    _log(FAN_OUT_NODE, "start", enabled_agents=[a.value for a in ENABLED_ANALYST_AGENTS])

    send_list: list[Send] = []

    for agent in ENABLED_ANALYST_AGENTS:
        
        node_name = get_agent_node_name(agent_name=agent)
        send_list.append(
            Send(
                node_name,
                state
            )
        )
        _log(FAN_OUT_NODE, "dispatch", target=node_name)
        
    return send_list
            
def tech_reviewer_node(state: DueDiligence_State):
    
    tech_reviewer = TechReviewer_Agent(framework=Framework.LANG_GRAPH)
    node = TECH_REVIEWER_NODE
    _log(node, "start", company=state.company_info.company_name if state.company_info else None, has_profile=state.company_profile is not None)
    t0 = time.perf_counter()
    tech_reviewer.build_tasks(state.topic_for("tech"))
    
    agent = tech_reviewer.get_agent()
    _log(node, "agent built", tools=list(tech_reviewer.tools), topic_chars=len(state.topic_for("tech")))
    
    result = agent.invoke(
                {
                    "messages": tech_reviewer.get_tasks()
                },
            )
    _log(node, "end", elapsed_s=round(time.perf_counter() - t0, 2))
    return {"company_profile": CompanyProfile(tech=[result["messages"][-1].content])}

def commercial_analyst_node(state: DueDiligence_State):
    
    commercial_analyst = CommercialAnalyst_Agent(framework=Framework.LANG_GRAPH)
    node = COMMERCIAL_ANALYST_NODE
    _log(node, "start", company=state.company_info.company_name if state.company_info else None, has_profile=state.company_profile is not None)
    t0 = time.perf_counter()
    commercial_analyst.build_tasks(state.topic_for("commercial"))
    
    agent = commercial_analyst.get_agent()
    _log(node, "agent built", tools=list(commercial_analyst.tools), topic_chars=len(state.topic_for("commercial")))

    result = agent.invoke(
                {
                    "messages": commercial_analyst.get_tasks()
                },
            )
    _log(node, "end", elapsed_s=round(time.perf_counter() - t0, 2))
    return {"company_profile": CompanyProfile(commercial=[result["messages"][-1].content])}

def team_analyst_node(state: DueDiligence_State):
    
    team_analyst = TeamAnalyst_Agent(framework=Framework.LANG_GRAPH)    
    node = TEAM_ANALYST_NODE
    _log(node, "start", company=state.company_info.company_name if state.company_info else None, has_profile=state.company_profile is not None)
    t0 = time.perf_counter()
    team_analyst.build_tasks(state.topic_for("team"))
    
    agent = team_analyst.get_agent()
    _log(node, "agent built", tools=list(team_analyst.tools), topic_chars=len(state.topic_for("team")))

    result = agent.invoke(
                {
                    "messages": team_analyst.get_tasks()
                },
            )
    _log(node, "end", elapsed_s=round(time.perf_counter() - t0, 2))
    return {"company_profile": CompanyProfile(team=[result["messages"][-1].content])}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/analysis_review.py
import asyncio

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.agents.structured_output import ToolStrategy

from due_diligence_mas.mas_definition.classes.company import AnalysisReview
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.langgraph import SUPERVISOR_NODE
from due_diligence_mas.mas_definition.constants.review import ANALYSIS_REVIEW_CRITERIA, TASK_CREATION_INSTRUCTIONS
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State, SupervisionState
from due_diligence_mas.mas_definition.providers.llm_provider import LLM_PROVIDER
from due_diligence_mas.mas_definition.tools.mcp_task import get_task_tools
from due_diligence_mas.mas_definition.utils.backstory_builder import backstory_builder
from due_diligence_mas.utils.logger import log_node as _log

MAX_TASKS = 2


def review_analysis(state: DueDiligence_State):
    """Completeness gate: an agent scores the aggregated analysis from 0 to 10 before the debate and, when it decides follow-up work is needed, creates tasks through the task manager MCP tools. MCP tools are async only, the graph is sync."""
    generic = "\n".join(state.company_profile.generic or []) if state.company_profile else ""
    content = f"COMPANY: {state.describe()}\n\nPROFILE:\n{generic}\n\nAGGREGATED ANALYSIS:\n{state.analysis_results}"
    review = asyncio.run(_review_analysis_agent(content))
    _log(SUPERVISOR_NODE, "analysis review", score=review.score, missing=len(review.missing))
    return {"supervision": SupervisionState(analysis_review=review)}


async def _review_analysis_agent(content: str) -> AnalysisReview:
    """Run the review agent with the task tools. Hard cap of one task per topic; no checkpointer, so it does not inherit the graph's sync SqliteSaver, which cannot run under ainvoke."""
    agent = create_agent(
        model=LLM_PROVIDER().get_chat_model(),
        tools=await get_task_tools(),
        system_prompt=backstory_builder(AgentName.ANALYST_SUPERVISOR.value),
        response_format=ToolStrategy(AnalysisReview),
        middleware=[ToolCallLimitMiddleware(tool_name="create_task", run_limit=MAX_TASKS)],
        checkpointer=False,
    )
    result = await agent.ainvoke({"messages": [{"role": "user", "content": f"{ANALYSIS_REVIEW_CRITERIA}\n\n{TASK_CREATION_INSTRUCTIONS}\n\n{content}"}]})
    _log(SUPERVISOR_NODE, "analysis review agent", tools_called=[m.name for m in result["messages"] if getattr(m, "type", "") == "tool"])
    return result["structured_response"]

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/human_in_the_loop.py
from typing import Optional

from due_diligence_mas.mas_definition.classes.company import AnalysisReview
from due_diligence_mas.mas_definition.constants.langgraph import MEMO_WRITER_NODE
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from due_diligence_mas.mas_definition.langgraph.node.supervisor import debate_entry_node
from langgraph.types import interrupt
from rich.console import Console
from rich.markdown import Markdown


def _print_preview(analysis: str, review: Optional[AnalysisReview]):
    """Print the supervisor's analysis review (score, missing items) and the analysis as formatted markdown."""
    console = Console()
    if review:
        missing = "\n".join(f"- {m}" for m in review.missing) or "- none"
        console.print(Markdown(f"## Supervisor review: completeness score {review.score}/10\n\n{review.summary}\n\n**Missing:**\n{missing}\n\n---"))
    console.print(Markdown(analysis or "_No analysis available_"))


def human_review_node(state: DueDiligence_State):
    """Pause the graph and wait for the human decision on the analysis; resumes with the decision as value."""
    review = state.supervision.analysis_review
    _print_preview(state.analysis_results, review)

    human_decision = interrupt({
        "message": "The analysis is complete. Review the results and decide how to proceed.",
        "analysis_preview": state.analysis_results,
        "analysis_review": review.model_dump() if review else None,
        "options": {
            "Proceed": "Approve and proceed with the debate",
            "End": "Skip the debate and write the memo from the analysis only",
        },
    })
    return {"human_decision": str(human_decision).strip().lower()}


def preview_and_confirm_router(state: DueDiligence_State):
    """Proceed goes on to the debate, anything else goes straight to the memo writer."""
    if state.human_decision == "proceed":
        return debate_entry_node()
    return MEMO_WRITER_NODE

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/debate.py
import time

from due_diligence_mas.mas_definition.agents.debate.debate_formatter import DebateFormatter_Agent
from due_diligence_mas.mas_definition.agents.debate.debate_judge import DebateJudge_Agent
from due_diligence_mas.mas_definition.agents.debate.opportunity_analyst import OpportunityAnalyst_Agent
from due_diligence_mas.mas_definition.agents.debate.risk_analyst import RiskAnalyst_Agent
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from due_diligence_mas.mas_definition.utils.agent_names import get_agent_node_name
from due_diligence_mas.utils.logger import log_node as _log


def _run_debate_agent(agent_cls, topic: str, context: list[str]) -> str:
    """Build the agent and its task from the topic and the previous debate outputs, and return its report."""
    node = get_agent_node_name(agent_cls.name)
    _log(node, "start", topic_chars=len(topic), context_items=len(context))
    t0 = time.perf_counter()

    debate_agent = agent_cls(framework=Framework.LANG_GRAPH)
    debate_agent.build_tasks(topic, context)

    agent = debate_agent.get_agent()
    _log(node, "agent built", tools=list(debate_agent.tools))

    result = agent.invoke(
                {
                    "messages": debate_agent.get_tasks()
                },
            )
    report = result["messages"][-1].content

    _log(node, "end", elapsed_s=round(time.perf_counter() - t0, 2))
    return report


def debate_formatter_node(state: DueDiligence_State):
    
    pack = _run_debate_agent(DebateFormatter_Agent, state.analysis_results or "", [])
    return {"debate_pack": pack}


def _debate_context(state: DueDiligence_State) -> str:
    """Debate input: the formatter pack, or the analysis results when the formatter is skipped."""
    return state.debate_pack or state.analysis_results or ""


def opportunity_analyst_node(state: DueDiligence_State):
    
    pro_case = _run_debate_agent(OpportunityAnalyst_Agent, "", [_debate_context(state)])
    return {"pro_case": pro_case}


def risk_analyst_node(state: DueDiligence_State):
    
    contro_case = _run_debate_agent(RiskAnalyst_Agent, "", [_debate_context(state), state.pro_case])
    return {"contro_case": contro_case}


def debate_judge_node(state: DueDiligence_State):
    
    judgement = _run_debate_agent(DebateJudge_Agent, "", [state.pro_case, state.contro_case])
    return {"debate_results": judgement}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/node/writer.py
import time

from due_diligence_mas.mas_definition.agents.writer.memo_writer import MemoWriter_Agent
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.constants.langgraph import ANALYSIS_AGGREGATOR_NODE, MEMO_WRITER_NODE
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from due_diligence_mas.utils.logger import log_node as _log


def _run_writer_agent(node: str, task: str, topic: str) -> str:
    """Build the memo writer for the given task ("aggregation" or "memo") and return its output."""
    _log(node, "start", task=task, topic_chars=len(topic))
    t0 = time.perf_counter()

    writer = MemoWriter_Agent(framework=Framework.LANG_GRAPH, task=task)
    writer.build_tasks(topic, key=task)

    agent = writer.get_agent()
    _log(node, "agent built", tools=list(writer.tools))

    result = agent.invoke(
                {
                    "messages": writer.get_tasks()
                },
            )
    output = result["messages"][-1].content

    _log(node, "end", elapsed_s=round(time.perf_counter() - t0, 2))
    return output


def analysis_aggregator_node(state: DueDiligence_State):
    """Consolidate the specialists' reports into a single analysis."""
    profile = state.company_profile
    reports = {
        "TECHNOLOGY": profile.tech if profile else None,
        "COMMERCIAL": profile.commercial if profile else None,
        "TEAM": profile.team if profile else None,
    }
    topic = "\n\n".join(f"{name} REPORT:\n" + "\n".join(entries or ["Not available"]) for name, entries in reports.items())

    return {"analysis_results": _run_writer_agent(ANALYSIS_AGGREGATOR_NODE, "aggregation", topic)}


def memo_writer_node(state: DueDiligence_State):
    """Write the final memo from the analysis and the debate (if performed)."""
    topic = (
        f"ANALYSIS:\n{state.analysis_results}\n\n"
        f"DEBATE:\n{state.debate_results or 'Debate not performed'}"
    )
    
    final_report =_run_writer_agent(MEMO_WRITER_NODE, "memo", topic)

    return {"final_report": final_report}

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/mas_definition/langgraph/graph/due_diligence.py
import sqlite3
import uuid
from due_diligence_mas.mas_definition.classes import company as company_classes
from due_diligence_mas.mas_definition.classes.company import CompanyInfo
from due_diligence_mas.mas_definition.langgraph.classes import shared_state
from due_diligence_mas.mas_definition.constants.agents_definition import AgentName
from due_diligence_mas.mas_definition.constants.langgraph import (
    ANALYSIS_AGGREGATOR_NODE, ANALYST_AGENTS, DEBATE_FORMATTER_NODE, DEBATE_JUDGE_NODE, ENABLED_ANALYST_AGENTS, FAN_OUT_NODE, HUMAN_REVIEW_NODE,
    MEMO_WRITER_NODE, OPPORTUNITY_ANALYST_NODE, PROFILER_NODE, REQUEST_HANDLER_NODE, RISK_ANALYST_NODE, SUPERVISOR_NODE, skip_debate_formatter,
)
from due_diligence_mas.mas_definition.langgraph.classes.shared_state import DueDiligence_State
from due_diligence_mas.mas_definition.langgraph.node.analysts import analyst_fan_out_node, commercial_analyst_node, team_analyst_node, tech_reviewer_node
from due_diligence_mas.mas_definition.langgraph.node.debate import debate_formatter_node, debate_judge_node, opportunity_analyst_node, risk_analyst_node
from due_diligence_mas.mas_definition.langgraph.node.profiler import company_profiler_node
from due_diligence_mas.mas_definition.langgraph.node.supervisor import debate_entry_node, supervisor_node, supervisor_router
from due_diligence_mas.mas_definition.langgraph.node.human_in_the_loop import human_review_node, preview_and_confirm_router
from due_diligence_mas.mas_definition.langgraph.node.request_handler import request_handler_node
from due_diligence_mas.mas_definition.langgraph.node.writer import analysis_aggregator_node, memo_writer_node
from due_diligence_mas.mas_definition.utils.agent_names import get_agent_node_name as node_name
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from dotenv import load_dotenv
from due_diligence_mas.mas_definition.constants.common import Framework
from due_diligence_mas.mas_definition.utils.execution_db import (
    ExecutionStatus, create_execution, db_path, delete_tracker, get_execution, load_tracker, save_tracker, update_execution,
)
from due_diligence_mas.mas_definition.utils.usage_tracker import UsageTracker
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from langgraph.checkpoint.sqlite import SqliteSaver
from pydantic import BaseModel


AGENT_NODES = {
    AgentName.TECH_REVIEWER: tech_reviewer_node,
    AgentName.COMMERCIAL_ANALYST: commercial_analyst_node,
    AgentName.TEAM_ANALYST: team_analyst_node,
    AgentName.DEBATE_FORMATTER: debate_formatter_node,
    AgentName.OPPORTUNITY_ANALYST: opportunity_analyst_node,
    AgentName.RISK_ANALYST: risk_analyst_node,
    AgentName.DEBATE_JUDGE: debate_judge_node,
}


def _pydantic_models(*modules) -> list[tuple[str, str]]:
    """(module, class) of the pydantic models defined in the modules: the types allowed in the checkpoints."""
    return [
        (module.__name__, name)
        for module in modules
        for name, cls in vars(module).items()
        if isinstance(cls, type) and issubclass(cls, BaseModel) and cls.__module__ == module.__name__
    ]


STATE_MODELS = _pydantic_models(company_classes, shared_state)


class DueDiligence_Graph():
    
    def __init__(self):
        
        load_dotenv()
        
        self.graph_builder = StateGraph(DueDiligence_State)
        
        self._build_nodes()
        self._build_edges()
        self._compile_grap()
        
    def _build_nodes(self):
        
        self.graph_builder.add_node(REQUEST_HANDLER_NODE, request_handler_node)
        self.graph_builder.add_node(PROFILER_NODE, company_profiler_node)
        self.graph_builder.add_node(SUPERVISOR_NODE, supervisor_node)
        self.graph_builder.add_node(HUMAN_REVIEW_NODE, human_review_node)
        self.graph_builder.add_node(FAN_OUT_NODE, lambda state: {})
        self.graph_builder.add_node(ANALYSIS_AGGREGATOR_NODE, analysis_aggregator_node)
        
        self.graph_builder.add_node(MEMO_WRITER_NODE, memo_writer_node)

        for agent, function in AGENT_NODES.items():
            if agent == AgentName.DEBATE_FORMATTER and skip_debate_formatter():
                continue
            self.graph_builder.add_node(node_name(agent), function)

    def _build_edges(self):
        """Wire the graph: profiler <-> supervisor loop, human review, analyst fan-out/join, optional debate (formatter, PRO, CONTRO, judge) and memo writer."""
        self.graph_builder.add_edge(START, REQUEST_HANDLER_NODE)
        self.graph_builder.add_edge(REQUEST_HANDLER_NODE, PROFILER_NODE)
        self.graph_builder.add_edge(PROFILER_NODE, SUPERVISOR_NODE)

        self.graph_builder.add_conditional_edges(SUPERVISOR_NODE, supervisor_router, [PROFILER_NODE, FAN_OUT_NODE, HUMAN_REVIEW_NODE])
        self.graph_builder.add_conditional_edges(HUMAN_REVIEW_NODE, preview_and_confirm_router, [debate_entry_node(), MEMO_WRITER_NODE])
        self.graph_builder.add_conditional_edges(FAN_OUT_NODE, analyst_fan_out_node, [node_name(a) for a in ANALYST_AGENTS])

        self.graph_builder.add_edge([node_name(a) for a in ENABLED_ANALYST_AGENTS], ANALYSIS_AGGREGATOR_NODE)
        self.graph_builder.add_edge(ANALYSIS_AGGREGATOR_NODE, SUPERVISOR_NODE)
        
        if not skip_debate_formatter():
            self.graph_builder.add_edge(DEBATE_FORMATTER_NODE, OPPORTUNITY_ANALYST_NODE)
        self.graph_builder.add_edge(OPPORTUNITY_ANALYST_NODE, RISK_ANALYST_NODE)
        self.graph_builder.add_edge(RISK_ANALYST_NODE, DEBATE_JUDGE_NODE)
        self.graph_builder.add_edge(DEBATE_JUDGE_NODE, MEMO_WRITER_NODE)
        
        self.graph_builder.add_edge(MEMO_WRITER_NODE, END)

        
    def _compile_grap(self):
        """Compile the graph. State checkpoints share the executions db, so a paused execution survives a restart."""
        serializer = JsonPlusSerializer(allowed_msgpack_modules=STATE_MODELS)
        checkpointer = SqliteSaver(sqlite3.connect(db_path(), check_same_thread=False), serde=serializer)

        self.compiled_graph = self.graph_builder.compile(checkpointer=checkpointer)

    def run(self, company_info: CompanyInfo = None, thread_id: str = None) -> dict:
        """Start the graph: it runs until the human review (interrupt) or the end."""
        if thread_id and self._check_status(thread_id) is not None:
            raise ValueError("Ongoing or completed execution.")
        create_execution(thread_id, Framework.LANG_GRAPH.value, company_info.company_name if company_info else None)

        return self._execute(DueDiligence_State(company_info=company_info), thread_id, UsageTracker())

    def resume(self, decision, thread_id: str) -> dict:       
        """Resume the graph after the human review with the decision (Proceed / End)."""
        
        if self._check_status(thread_id, ExecutionStatus.AWAITING_REVIEW.value) == False:
            raise ValueError("The execution is not interrupted")
        
        snapshot = load_tracker(thread_id)
        tracker = UsageTracker.restore(snapshot) if snapshot else UsageTracker()
        update_execution(thread_id, status=ExecutionStatus.RUNNING, pending_review=None)

        return self._execute(Command(resume=decision), thread_id, tracker)

    def _execute(self, graph_input, thread_id: str, tracker: UsageTracker) -> dict:
        """Stream the graph keeping the execution row up to date, then close it as awaiting review or completed. The human wait between run() and resume() is outside measure(), so it is not counted."""
        config = {"configurable": {"thread_id": thread_id}, "callbacks": [tracker]}
        interrupt_payload = None

        try:
            with tracker.measure():
                for chunk in self.compiled_graph.stream(graph_input, config, stream_mode="updates"):
                    for node, update in chunk.items():
                        if node == "__interrupt__":
                            interrupt_payload = update[0].value
                        else:
                            update_execution(thread_id, current_node=node)

            state = self.compiled_graph.get_state(config).values
            company = state["company_info"].company_name if state.get("company_info") else None

            if interrupt_payload is not None:
                return self._pause(thread_id, tracker, company, interrupt_payload)
            return self._complete(thread_id, tracker, company, state.get("final_report"))
        except Exception:
            update_execution(thread_id, status=ExecutionStatus.FAILED, metrics=tracker.metrics())
            raise

    def _pause(self, thread_id: str, tracker: UsageTracker, company: str, interrupt_payload: dict) -> dict:
        """Waiting for the human review: keep the usage totals for the resume."""
        save_tracker(thread_id, tracker.snapshot())
        update_execution(
            thread_id, 
            status=ExecutionStatus.AWAITING_REVIEW, 
            current_node=HUMAN_REVIEW_NODE, 
            company_name=company,
            metrics=tracker.metrics(), 
            pending_review=interrupt_payload,
        )
        return {"sessionId": thread_id, "status": ExecutionStatus.AWAITING_REVIEW.value, "pending_review": interrupt_payload}

    def _complete(self, thread_id: str, tracker: UsageTracker, company: str, final_report: str) -> dict:
        """Store the memo and the metrics in the db row."""
        fields = {"status": ExecutionStatus.COMPLETED, "current_node": None, "company_name": company, "metrics": tracker.metrics()}
        outcome = {"sessionId": thread_id, "status": ExecutionStatus.COMPLETED.value, "final_report": final_report}
        if final_report:
            fields["memo_markdown"] = final_report

        update_execution(thread_id, **fields)
        delete_tracker(thread_id)
        return outcome

    @staticmethod
    def _check_status(thread_id: str, expected_status: str = None) -> bool | None:
        """ 
            Check existance and status of an execution
            
            Returns:
                None if the execution doesn't exists
                True if the status is correct
                False if the status is wrong
        """
        execution = get_execution(thread_id)
        if execution is None:
            return None
        else:
            return True if execution["status"] == expected_status else False

def main():
    thread_id = str(uuid.uuid4())
    graph = DueDiligence_Graph()
    outcome = graph.run(thread_id=thread_id)
    while outcome["status"] == ExecutionStatus.AWAITING_REVIEW.value:
        outcome = graph.resume(input("Proceed / End > "), thread_id)


if __name__ == "__main__":
    main()

## 6. Servers

- The FastAPI gateways come from the repo (`api_gateway/`). The repo starts them with `uvicorn.run(..., reload=True)`. Here each one runs in a **thread** with `uvicorn.Server`, without reload.
- The MCP server runs as a **subprocess** over `streamable-http`. The transport is read at import, and `mcp.run` blocks.
- The agents do not use this HTTP MCP server. They start their own stdio subprocess (section 3).

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/api_gateway/due_diligence_crewai.py
from due_diligence_mas.mas_definition.classes.company import CompanyInfo
from due_diligence_mas.mas_definition.crewai_fr.flows.due_diligence import resume_due_diligence_flow, run_due_diligence_flow
from due_diligence_mas.mas_definition.utils.execution_db import ExecutionStatus, get_execution, update_execution
from due_diligence_mas.utils.session_manager import create_session_id
from fastapi.responses import PlainTextResponse
from fastapi import BackgroundTasks, FastAPI, HTTPException, Response, Cookie
import os
import uvicorn
from enum import Enum

from pydantic import BaseModel
from due_diligence_mas.utils.logger import setup_logger

logger = setup_logger("ca_api_gateway")

class InferenceRequest(BaseModel):
    company_info: CompanyInfo

class Decision(Enum):
    PROCEED = "proceed"
    END = "end"
class ResumeRequest(BaseModel):
    decision: Decision

flow_gateway = FastAPI()


@flow_gateway.get("/health")
async def health():
    return {"status": "available"}
    
def _run_in_background(company_info: CompanyInfo, session_id: str):
    try:
        run_due_diligence_flow(company_info, session_id)
    except Exception as e:
        logger.exception(f"Background run failed (sessionId: {session_id}): {e}")

def _resume_in_background(decision: str, session_id: str):
    try:
        resume_due_diligence_flow(decision, session_id)
    except Exception as e:
        logger.error(f"Background resume failed (sessionId: {session_id}): {e}")

@flow_gateway.post('/inference')
async def run_graph(
    payload: InferenceRequest,
    response: Response,
    background_tasks: BackgroundTasks,
    sessionId = Cookie(None)
):
    """ 
        Always create a new session
    """
    try:
        sessionId = create_session_id()
            
        background_tasks.add_task(_run_in_background, payload.company_info, sessionId)
  
        """ 
            Set sessionId cookie that will be used in next iterations 
        """  
        response.set_cookie(key="sessionId", value=sessionId)

        return {"status": "initialized"}
    
    except Exception as e:
                
        error_message = getattr(e, "message", None) or str(e)
        status_code = getattr(e, "status_code", 500)
        logger.error(f"Inference failed (sessionId: {sessionId}, status: {status_code}): {error_message}")
        raise HTTPException(status_code=status_code, detail=f"Sorry, something went wrong. - Details: {error_message}")
   
@flow_gateway.post('/resume')
async def resume_after_interrupt(
    payload: ResumeRequest,
    response: Response,
    background_tasks: BackgroundTasks,
    sessionId = Cookie(None)
):
    
    try:
    
        if (sessionId is None):
            raise ValueError("SessionId not provided")

        execution = get_execution(sessionId)
        if execution is None or execution["status"] != ExecutionStatus.AWAITING_REVIEW.value:
            raise HTTPException(status_code=409, detail="No execution awaiting review to resume for this session")

        background_tasks.add_task(_resume_in_background, payload.decision.value, sessionId)

        return {"sessionId": sessionId, "status": "resumed"}
    
    except HTTPException:
        raise
    except Exception as e:
                
        error_message = getattr(e, "message", None) or str(e)
        status_code = getattr(e, "status_code", 500)
        logger.error(f"Resume failed (sessionId: {sessionId}, status: {status_code}): {error_message}")
        raise HTTPException(status_code=status_code, detail=f"Sorry, something went wrong. - Details: {error_message}")     
    
@flow_gateway.get('/execution')
async def get_execution_status(sessionId = Cookie(None)):
    if sessionId is None:
        raise HTTPException(status_code=400, detail="SessionId not provided")
    execution = get_execution(sessionId)
    if execution is None:
        raise HTTPException(status_code=404, detail=f"Execution {sessionId} not found")
    return execution

@flow_gateway.get('/memo', response_class=PlainTextResponse)
async def get_memo(sessionId = Cookie(None)):
    if sessionId is None:
        raise HTTPException(status_code=400, detail="SessionId not provided")
    execution = get_execution(sessionId)
    if execution is None:
        raise HTTPException(status_code=404, detail=f"Execution {sessionId} not found")
    if not execution["memo_markdown"]:
        raise HTTPException(status_code=409, detail=f"Memo not available yet (status: {execution['status']})")
    return PlainTextResponse(execution["memo_markdown"], media_type="text/markdown; charset=utf-8")

""" @flow_gateway.get('/session/{session_id}/data')
def get_session_data(session_id: str):
    
    try:
        return cc_agent.export_session_data(sessionId=session_id)
    
    except Exception as e:
        logger.error(f"Failed to export session data (session_id: {session_id}): {e}")
        return e


     """
     
     
def init_gateway():    
    
    host = os.getenv("AGENT_GATEWAY_HOST", "127.0.0.1")
    port = int(os.getenv("AGENT_GATEWAY_PORT", "8100"))

    try:
        
        uvicorn.run(
            "due_diligence_mas.api_gateway.due_diligence_crewai:flow_gateway",
            host=host,
            port=port,
            reload=True,
        )
        
    except Exception as e:
        logger.error(f"Error initializing Agent gateway: {e}")
        print(f"    Error initialing Agent gateway - Details: {e}")

In [ ]:
%%writefile /content/Due-Diligence-MAS/src/due_diligence_mas/api_gateway/due_diligence_langgraph.py
from due_diligence_mas.mas_definition.classes.company import CompanyInfo
from due_diligence_mas.mas_definition.langgraph.graph.due_diligence import DueDiligence_Graph
from due_diligence_mas.mas_definition.utils.execution_db import ExecutionStatus, get_execution, update_execution
from due_diligence_mas.utils.session_manager import create_session_id
from fastapi.responses import PlainTextResponse
from fastapi import BackgroundTasks, FastAPI, HTTPException, Response, Cookie
import os
import uvicorn
from enum import Enum

from pydantic import BaseModel
from due_diligence_mas.utils.logger import setup_logger

logger = setup_logger("lg_api_gateway")

class InferenceRequest(BaseModel):
    company_info: CompanyInfo

class Decision(Enum):
    PROCEED = "proceed"
    END = "end"
class ResumeRequest(BaseModel):
    decision: Decision

graph_gateway = FastAPI()

due_diligence_graph = DueDiligence_Graph()

@graph_gateway.get("/health")
async def health():
    return {"status": "available"}
    
def _run_in_background(company_info: CompanyInfo, session_id: str):
    """Run the graph after the response is sent; follow it with GET /execution. The graph already marks the execution row as failed."""
    try:
        due_diligence_graph.run(company_info, session_id)
    except Exception as e:
        logger.exception(f"Background run failed (sessionId: {session_id}): {e}")

def _resume_in_background(decision: str, session_id: str):
    try:
        due_diligence_graph.resume(decision, session_id)
    except Exception as e:
        logger.error(f"Background resume failed (sessionId: {session_id}): {e}")

@graph_gateway.post('/inference')
async def run_graph(
    payload: InferenceRequest,
    response: Response,
    background_tasks: BackgroundTasks,
    sessionId = Cookie(None)
):
    """ 
        Always create a new session
    """
    try:
        sessionId = create_session_id()
            
        background_tasks.add_task(_run_in_background, payload.company_info, sessionId)
  
        """ 
            Set sessionId cookie that will be used in next iterations 
        """  
        response.set_cookie(key="sessionId", value=sessionId)

        return {"status": "initialized"}
    
    except Exception as e:
                
        error_message = getattr(e, "message", None) or str(e)
        status_code = getattr(e, "status_code", 500)
        logger.error(f"Inference failed (sessionId: {sessionId}, status: {status_code}): {error_message}")
        raise HTTPException(status_code=status_code, detail=f"Sorry, something went wrong. - Details: {error_message}")
   
@graph_gateway.post('/resume')
async def resume_after_interrupt(
    payload: ResumeRequest,
    response: Response,
    background_tasks: BackgroundTasks,
    sessionId = Cookie(None)
):
    
    try:
    
        if (sessionId is None):
            raise ValueError("SessionId not provided")

        execution = get_execution(sessionId)
        if execution is None or execution["status"] != ExecutionStatus.AWAITING_REVIEW.value:
            raise HTTPException(status_code=409, detail="No execution awaiting review to resume for this session")

        background_tasks.add_task(_resume_in_background, payload.decision.value, sessionId)

        return {"sessionId": sessionId, "status": "resumed"}
    
    except HTTPException:
        raise
    except Exception as e:
                
        error_message = getattr(e, "message", None) or str(e)
        status_code = getattr(e, "status_code", 500)
        logger.error(f"Resume failed (sessionId: {sessionId}, status: {status_code}): {error_message}")
        raise HTTPException(status_code=status_code, detail=f"Sorry, something went wrong. - Details: {error_message}")     
    
@graph_gateway.get('/execution')
async def get_execution_status(sessionId = Cookie(None)):
    if sessionId is None:
        raise HTTPException(status_code=400, detail="SessionId not provided")
    execution = get_execution(sessionId)
    if execution is None:
        raise HTTPException(status_code=404, detail=f"Execution {sessionId} not found")
    return execution

@graph_gateway.get('/memo', response_class=PlainTextResponse)
async def get_memo(sessionId = Cookie(None)):
    if sessionId is None:
        raise HTTPException(status_code=400, detail="SessionId not provided")
    execution = get_execution(sessionId)
    if execution is None:
        raise HTTPException(status_code=404, detail=f"Execution {sessionId} not found")
    if not execution["memo_markdown"]:
        raise HTTPException(status_code=409, detail=f"Memo not available yet (status: {execution['status']})")
    return PlainTextResponse(execution["memo_markdown"], media_type="text/markdown; charset=utf-8")

""" @graph_gateway.get('/session/{session_id}/data')
def get_session_data(session_id: str):
    
    try:
        return cc_agent.export_session_data(sessionId=session_id)
    
    except Exception as e:
        logger.error(f"Failed to export session data (session_id: {session_id}): {e}")
        return e


     """
     
     
def init_gateway():    
    
    host = os.getenv("AGENT_GATEWAY_HOST", "127.0.0.1")
    port = int(os.getenv("AGENT_GATEWAY_PORT", "8000"))

    try:
        
        uvicorn.run(
            "due_diligence_mas.api_gateway.due_diligence_langgraph:graph_gateway",
            host=host,
            port=port,
            reload=True,
        )
        
    except Exception as e:
        logger.error(f"Error initializing Agent gateway: {e}")
        print(f"    Error initialing Agent gateway - Details: {e}")

### Start the servers

In [ ]:
import socket
import subprocess
import time

import requests

HOST = "127.0.0.1"
PORTS = {"langgraph": 8000, "crewai": 8100}
MCP_PORT = 8200
PROCESSES = {}

# Each server is a subprocess: its logs go to ROOT/log/<name>.log and not to the notebook.
# start_new_session: stopping a cell (SIGINT) must not stop the servers.
COMMANDS = {
    "langgraph": [sys.executable, "-m", "uvicorn", "due_diligence_mas.api_gateway.due_diligence_langgraph:graph_gateway", "--host", HOST, "--port", str(PORTS["langgraph"])],
    "crewai": [sys.executable, "-m", "uvicorn", "due_diligence_mas.api_gateway.due_diligence_crewai:flow_gateway", "--host", HOST, "--port", str(PORTS["crewai"])],
    "mcp": [sys.executable, "-c", "from due_diligence_mas.mcp.task_manager import main; main()"],
}
ENVS = {"mcp": {"MCP_TRANSPORT": "streamable-http", "MCP_HOST": HOST, "MCP_PORT": str(MCP_PORT)}}
SERVER_PORTS = {**PORTS, "mcp": MCP_PORT}

(ROOT / "log").mkdir(exist_ok=True)
for name, command in COMMANDS.items():
    log_path = ROOT / "log" / f"{name}_server.log"
    PROCESSES[name] = subprocess.Popen(
        command,
        env={**os.environ, **ENVS.get(name, {})},
        stdout=open(log_path, "w"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(120):
        if PROCESSES[name].poll() is not None:
            raise RuntimeError(f"{name} server exited, see {log_path}")
        try:
            socket.create_connection((HOST, SERVER_PORTS[name]), timeout=1).close()
            break
        except OSError:
            time.sleep(0.5)
    else:
        raise RuntimeError(f"{name} server did not start, see {log_path}")
print({name: f"http://{HOST}:{port}" for name, port in PORTS.items()}, f"mcp=http://{HOST}:{MCP_PORT}/mcp")

Health checks.

In [ ]:
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

for name, port in PORTS.items():
    print(name, requests.get(f"http://{HOST}:{port}/health", timeout=5).json())


async def list_mcp_tools():
    async with streamablehttp_client(f"http://{HOST}:{MCP_PORT}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return [t.name for t in (await session.list_tools()).tools]


print("mcp", await asyncio.wait_for(list_mcp_tools(), timeout=30))

### Run a due diligence

WARNING: this uses your LLM and Tavily credits.

1. Start the run. 2. Poll until the status changes. 3. Resume after the human review.

In [ ]:
from IPython.display import Markdown, display


def wait():
    """Print each status change; stop at awaiting_review, completed or failed. Show the memo when completed."""
    last = None
    while True:
        execution = requests.get(f"{BASE}/execution", cookies=COOKIES, timeout=30).json()
        state = (execution.get("status", "starting"), execution.get("current_node"), execution.get("current_crew"))
        if state != last:
            print(time.strftime("%H:%M:%S"), *state)
            last = state
        if state[0] in ("awaiting_review", "completed", "failed"):
            break
        time.sleep(15)
    if state[0] == "failed":
        print(f"See {ROOT / 'log' / (FRAMEWORK + '_server.log')}")
    if state[0] == "completed":
        display(Markdown(requests.get(f"{BASE}/memo", cookies=COOKIES, timeout=30).text))


FRAMEWORK = input("Framework (langgraph/crewai): ").strip().lower()
COMPANY = {
    "company_name": input("Company name: ").strip(),
    "company_site": input("Website: ").strip(),
    "company_sector": input("Sector: ").strip(),
    "additional_info": input("Additional info: ").strip(),
}

BASE = f"http://{HOST}:{PORTS[FRAMEWORK]}"
started = requests.post(f"{BASE}/inference", json={"company_info": COMPANY}, timeout=30)
started.raise_for_status()
COOKIES = {"sessionId": started.cookies["sessionId"]}
print(FRAMEWORK, COOKIES["sessionId"])

In [ ]:
wait()

In [ ]:
# Analysis preview shown to the human reviewer (status "awaiting_review").
pending = requests.get(f"{BASE}/execution", cookies=COOKIES, timeout=30).json().get("pending_review") or {}
display(Markdown(pending.get("analysis_preview") or pending.get("analysis") or "_No analysis preview available yet._"))

In [ ]:
# Only at status "awaiting_review".
decision = input("Decision (proceed/end): ").strip().lower()
requests.post(f"{BASE}/resume", json={"decision": decision}, cookies=COOKIES, timeout=30).raise_for_status()
time.sleep(3)

In [ ]:
wait()

In [ ]:
# Final memo (status "completed").
display(Markdown(requests.get(f"{BASE}/memo", cookies=COOKIES, timeout=30).text))

### Stop the servers

Run this cell when you are done. The servers also stop when the runtime recycles.

In [ ]:
for process in PROCESSES.values():
    process.terminate()